[Markdown Cell]

# Phase 5 / 06 B Distillation

Attach completed Phase 5 notebook 00 output, the **complete matching own Phase 1 foundation output**, and its original tokenizer.json/metadata.json if absent. Select **T4 x2**, Internet ON. Each notebook is independent and embeds its complete importable runtime.

For resume, attach the complete previous SAME Phase 5 output and set RESUME_RUN_DIR to the folder containing job.json and checkpoints. The launch cell inherits its learning choices, model, taxonomy and schedule. The raw .ipynb or observations.zip alone cannot resume. Prior Phase 2/3/4 task heads are not Phase 5 foundations.

[Markdown Cell]

## Fixed teacher discovery and an early input check

Your saved run stopped **before student training**: it found the B Phase 1 foundation, but
`teacher_candidates` was empty. Distillation needs a **completed Phase 5 A/C/D LSI teacher** as well.
The old error incorrectly said Phase 4. Phase 1 foundations, `.ipynb` files, observation ZIPs and
submission ZIPs are not complete trained LSI teacher outputs.

Attach these inputs before Run All:

1. Completed **Phase 5 notebook 00 output** with the prepared data and manifest.
2. The same **own B Phase 1 foundation output** already verified in your failed run, plus its matching
   tokenizer/metadata when required. There is no need to retrain that foundation.
3. The **complete saved output of Phase 5 A Full** (or completed A LoRMA/C Adapters/D Partial):
   `job.json`, `training_summary.json`, `submission_receipt.json`, `checkpoints/best.pt` and its JSON
   receipt, `code/lsi/`, `parent_source/`, and `predictions/train_logits.npy`, `train_order.json`,
   `dev_logits.npy`, `dev_order.json`. Keep the original complete output together.

Use Add Input → Notebooks → select the saved, executed teacher version. Set `TEACHER_RUN_DIR` to
its exact task root containing `job.json` (for example a folder ending in `P5U_01_A_Full`). Do not
guess a mount path from its display name. You may leave it None if only one compatible teacher is
attached; the preflight prints what it finds and any rejection reasons.

If only **training logits are missing**, export the completed teacher using its **exact matching**
training/export notebook: attach its full output and the same prepared data, set `RESUME_RUN_DIR`
to that teacher root, `MODE='export'`, `EXPORT_TRAIN_LOGITS=True`, and Run All. This performs inference
and export, not another epoch. Attach that saved export output here afterward.

The repaired launcher accepts both legacy and upgraded own Phase 5 teachers, while checking their
own source, checkpoint, logit, case-order and dataset receipts. It retains all 18 original student
runtime sources, architecture, experiment ID and learning settings. The known NumPy numeric-metadata
resume helper is applied separately to the workflow and spawned workers. KD is never silently
disabled or replaced with ordinary supervised training. Keep T4 x2.

This file matches the original B foundation used in your uploaded notebook. The original uploaded
file is preserved locally. Large train.bin/val.bin are not needed for this task.

[Markdown Cell]

## 1. Environment

In [ ]:
# [Code Cell]
import importlib.metadata as package_metadata
import json
import os
from pathlib import Path
import subprocess
import sys

requirements = {'tokenizers': 'tokenizers==0.23.2', 'numpy': 'numpy>=1.26,<3',
                'scikit-learn': 'scikit-learn>=1.4,<2', 'tqdm': 'tqdm>=4.66,<5',
                'huggingface-hub': 'huggingface_hub>=0.30,<2', 'ijson': 'ijson>=3.3,<4'}
install = []
for package, requirement in requirements.items():
    try:
        version = package_metadata.version(package)
        if '==' in requirement and version != requirement.split('==')[1]:
            install.append(requirement)
    except package_metadata.PackageNotFoundError:
        install.append(requirement)
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *install])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Dependencies ready; Kaggle PyTorch is preserved. Use a fresh kernel before Run All.')


[Markdown Cell]

## 2. Explicit settings and input paths

In [ ]:
# [Code Cell]
EXPERIMENT_ID='P5_06_B_Distillation'
FAMILY='b'
METHOD='kd'
RESUME_RUN_DIR=None
OUTPUT_ROOT='/kaggle/working/cs781_phase5/'+EXPERIMENT_ID
PREPARED_DIR=None
PARENT_RUN_DIR=None
TOKENIZER_PATH=None
METADATA_PATH=None
TEACHER_RUN_DIR=None  # Exact COMPLETE Phase 5 A/C/D task root containing job.json
EXPECTED_PARENT_SHA256=None
MAX_EPOCHS=4
TARGET_TOTAL_EPOCHS=None
TRAINING_MINUTES=180
CHUNK_TOKENS=256
CHUNK_MICRO_BATCH=8
BACKBONE_LR=5e-05
HEAD_LR=3e-4
HEAD_WIDTH=128
DROPOUT=.2
PROMPT_TOKENS=16
LOSS_TYPE='bce'
EXPORT_TRAIN_LOGITS=False
MODE='train'
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)


[Markdown Cell]

A fresh run starts from your actual own Phase 1 checkpoint. Set EXPECTED_PARENT_SHA256 if you want to pin one particular best snapshot; None permits a verified own continuation of the same source/data contract. Large train.bin/val.bin are unnecessary for Phase 5.

TARGET_TOTAL_EPOCHS is absolute: if a task completed E epochs, set it to E+1 to request one more epoch. Keep MAX_EPOCHS and the saved learning settings unchanged on resume. A time-limited run saves its next-update cursor and pauses; it must finish before producing a completed submission.

Chunk microbatch controls GPU scheduling, not data coverage or global case batch (2 GPUs × 1 case × 4 accumulation = 8). Lower it to 4 for a fresh run if memory is tight. Changing chunk length or learning settings is a separate experiment. Prompt tuning adds learned context without deleting source tokens.

[Markdown Cell]

## 3. Model and objective rationale

[Markdown Cell]

For each case, encode every document chunk, pool its real token features, then contextualize the chunk sequence. Each statute has a learned query over these chunks; different laws can use different evidence. The head returns one logit per statute, with independent probabilities $p_j=\sigma(z_j)$.

The supervised loss is a mean over labels and cases:
$$\ell=-\frac{1}{L}\sum_j [w_j y_j\log p_j+(1-y_j)\log(1-p_j)].$$
Positive weights use capped square roots of training-only negative/positive counts. The optional asymmetric trial downweights easy negatives. KD uses independent Bernoulli teacher targets at temperature T, never a single-label softmax. No fixed number of statutes is forced.

LoRMA uses the identity-inflated multiplicative form $(I+\alpha BA/r)W_0x+b$. Prompt B optimizes soft embeddings through its frozen encoder. C adapters learn bottleneck residuals; D updates its final layers; A full and B KD update their active foundation paths. All methods also learn the task head.

Validation determines the best snapshot and whether a global or shrunk label-wise threshold improves over 0.5 under grouped conditional checks. No private test answers or leaderboard scores enter that decision.

In [ ]:
# [Code Cell]
module_parts={}
def append_module(name, source):
    module_parts.setdefault(name, []).append(source)


[Markdown Cell]

### __init__.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('__init__.py', r'''"""Phase 5 own-foundation multi-label Legal Statute Identification."""
VERSION = "cs781-lsi-v1"
''')


[Markdown Cell]

### adaptation.py

Five distinct permitted techniques: full A, multiplicative LoRMA+ A, learned soft input prompts B, residual C adapters and final D layers. KD uses the smaller B encoder. The unused B decoder is frozen.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('adaptation.py', r'''# CELL: Five genuinely distinct active-foundation adaptation mechanisms
import math
import torch
from torch import nn
from torch.nn import functional as F


class LoRMA(nn.Module):
    """Frozen W0 plus multiplicative low-rank update: (I + alpha/r BA) W0 x + b."""
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank, base.out_features))
        self.b = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.linear(x, self.base.weight, None)
        output = hidden + self.scale * F.linear(F.linear(hidden, self.a), self.b)
        return output if self.base.bias is None else output + self.base.bias


class ResidualAdapter(nn.Module):
    """Frozen projection y plus a learned GELU bottleneck residual, initially zero."""
    def __init__(self, base: nn.Linear, width: int):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.down = nn.Linear(base.out_features, width)
        self.up = nn.Linear(width, base.out_features)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.base(x)
        return hidden + self.up(F.gelu(self.down(hidden)))


def adapt(backbone: nn.Module, cfg) -> list[str]:
    """Adapt active feature layers only; B's unused autoregressive decoder is excluded."""
    backbone.requires_grad_(False)
    if cfg.method in ('full', 'kd'):
        backbone.requires_grad_(True)
        if cfg.family == 'b':
            for module in (backbone.decoder, backbone.decoder_norm, backbone.target_positions):
                module.requires_grad_(False)
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('adaptation.py', r'''        return [name for name, p in backbone.named_parameters() if p.requires_grad]
    if cfg.method == 'prompt':
        width = backbone.shared_embedding.embedding_dim
        backbone.soft_prompt = nn.Parameter(torch.empty(cfg.prompt_tokens, width))
        nn.init.normal_(backbone.soft_prompt, std=.02)
        return ['soft_prompt']
    if cfg.method == 'partial':
        if cfg.partial_layers > len(backbone.layers):
            raise ValueError('partial_layers exceeds the actual D depth')
        for layer in backbone.layers[-cfg.partial_layers:]:
            layer.requires_grad_(True)
        backbone.final_norm.requires_grad_(True)
        return [name for name, p in backbone.named_parameters() if p.requires_grad]
    targets = list(backbone.named_modules())
    names = []
    for name, module in targets:
        replacement = None
        if cfg.method == 'lorma' and name.startswith('layers.') and isinstance(module, nn.Linear):
            replacement = LoRMA(module, cfg.rank, cfg.alpha)
        if (cfg.method == 'adapters' and name.startswith('stages.') and isinstance(module, nn.Linear)
                and name.endswith(('attention.out', 'ffn.down'))):
            replacement = ResidualAdapter(module, cfg.adapter_width)
        if replacement is not None:
            parent, _, child = name.rpartition('.')
            setattr(backbone.get_submodule(parent), child, replacement)
            names.append(name)
    if not names:
        raise ValueError('No intended backbone adaptation modules were found')
    return names
''')


[Markdown Cell]

### analysis.py

Dev-only frozen-weight ablations compare whole-document attention, whole-document mean and first/last sections, with coverage and length/error diagnostics. This isolates inference representation sensitivity; separately trained representation trials would still be needed for a causal training comparison.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('analysis.py', r'''# CELL: Optional frozen-head ablations with every document chunk retained
from dataclasses import replace
from pathlib import Path
import numpy as np
import torch
from .io_utils import atomic_json
from .predict import load_task,logits_partition,ordered_logits
from .metrics import probabilities,decisions,case_diagnostics,choose_threshold


def analyze(task_root: str, prepared_dir: str, output_root: str) -> dict:
    """Dev-only attention ablations; saved model tensors and train data are untouched.

    This measures sensitivity of one trained head. It is not a separately trained
    representation trial and cannot be counted as another adaptation technique.
    """
    output=Path(output_root);output.mkdir(parents=True,exist_ok=True)
    device=torch.device('cuda',0)
    model,job,cfg,_=load_task(Path(task_root),Path(prepared_dir),device,output/'load_cache')
    results={};original_weights=model.label_weights.detach().clone()
    for variant in ('trained_attention','shared_mean_all_chunks','no_statute_specific_attention'):
        model.cfg=replace(cfg,pooling='mean') if variant=='shared_mean_all_chunks' else cfg
        with torch.no_grad():
            model.label_weights.copy_(original_weights)
            if variant=='no_statute_specific_attention':model.label_weights.zero_()
        values=logits_partition(model,job,cfg,'dev',device)
        rows=job['cache']['cases']['dev'];p=probabilities(ordered_logits(values,rows))
        rule=choose_threshold(p,rows,cfg.calibrate_threshold,cfg.calibration_margin,cfg.seed)
        results[variant]={'raw':case_diagnostics([r['gold'] for r in rows],decisions(p,.5),rows),
            'calibrated':case_diagnostics([r['gold'] for r in rows],decisions(p,rule['thresholds']),rows),
            'calibration':rule,'all_document_chunks_retained':True}
    report={'status':'dev_ablation_complete','variants':results,'test_labels_used':False,
        'scope':'Frozen-weight inference ablations only; not separately trained methods or independent test estimates',
        'saved_checkpoints_modified':False}
    atomic_json(output/'representation_ablation_report.json',report)
    return report
''')


[Markdown Cell]

### compare.py

Check receipts, model/logit hashes and dev score reproduction. Compare six distinct experiments and copy the actual validation winner ZIP. This does not create a seventh adaptation method.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('compare.py', r'''# CELL: CPU-only evidence checks and validation selection across required methods
from pathlib import Path
import csv
import shutil
import numpy as np
from .io_utils import atomic_json, read_json, sha256_file
from .metrics import probabilities, decisions, metrics, prediction, choose_threshold, official
from .predict import package
from .parents import source_hashes


def compare(input_root: str, output: str, explicit_roots=None) -> dict:
    """Verify actual learned outputs and choose one by measured dev Macro-F1.

    This is selection, not an extra adaptation technique. Private test quality
    remains unknown. No label-dependent test feedback enters the selection.
    """
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    paths=[Path(p)/'submission_receipt.json' for p in explicit_roots] if explicit_roots else sorted(Path(input_root).rglob('submission_receipt.json'))
    rows=[];rejected=[]
    for path in paths:
        root=path.parent
        try:
            r=read_json(path)
            if r.get('version')!='lsi-output-v1':continue
            job=read_json(root/'job.json')
            if r['status']!='submission_verified' or read_json(root/'training_summary.json')['status']!='complete':raise ValueError('Incomplete task')
            if r['fingerprint']!=job['fingerprint'] or source_hashes(root/'code/lsi')!=job['runtime_source_sha256']:raise ValueError('Source/task identity mismatch')
            if sha256_file(root/'checkpoints/best.pt')!=r['checkpoint_sha256']:raise ValueError('Best checkpoint checksum mismatch')
            for name,expected in r['logit_artifacts'].items():
                if sha256_file(root/'predictions'/name)!=expected:raise ValueError('Logit/order checksum mismatch: '+name)
            truth=read_json(root/'predictions/dev_truth.json')
            order=read_json(root/'predictions/dev_order.json')
            if [x['id'] for x in truth]!=[x['id'] for x in order]:raise ValueError('Dev truth/order mismatch')
            scores=np.load(root/'predictions/dev_logits.npy',allow_pickle=False)
            value=metrics([x['gold'] for x in truth],decisions(probabilities(scores),r['threshold']))
            if abs(value['macro_f1']-r['dev_macro_f1'])>1e-9:raise ValueError('Receipt F1 does not reproduce')
            rows.append({'run_dir':str(root),'experiment_id':r['experiment_id'],'family':r['family'],'method':r['method'],
                'dev_macro_f1':value['macro_f1'],'micro_f1':value['micro_f1'],'subset_accuracy':value['subset_accuracy'],
                'prepared_identity':r['prepared_identity'],'label_count':r['label_count'],'checkpoint_sha256':r['checkpoint_sha256'],
                'trainable_parameters':r['parameter_report']['trainable_parameters'],'original_receipt':r})
        except (OSError,ValueError,KeyError) as error:rejected.append({'root':str(root),'error':str(error)})
    atomic_json(output/'rejected_inputs.json',rejected)
    if not rows:raise FileNotFoundError('Attach complete finished Phase 5 outputs with checkpoints, predictions/logits and receipts. No eligible LSI run found.')
    if len({(r['prepared_identity'],r['label_count']) for r in rows})!=1:raise ValueError('Cannot compare different prepared data/taxonomies')
    winner=max(rows,key=lambda r:(r['dev_macro_f1'],r['micro_f1'],r['experiment_id']))
    required={('a','full'),('a','lorma'),('b','prompt'),('c','adapters'),('d','partial'),('b','kd')}
    present={(r['family'],r['method']) for r in rows}
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('compare.py', r'''    source=Path(winner['run_dir']);receipt=winner['original_receipt']
    original_zip=source/Path(receipt['zip_path']).name
    if sha256_file(original_zip)!=receipt['zip_sha256']:raise ValueError('Winner ZIP checksum changed')
    target=output/'submission_Phase5_Validation_Winner.zip';shutil.copyfile(original_zip,target)
    report={'status':'validation_selection_verified','winner':{k:v for k,v in winner.items() if k!='original_receipt'},
        'runs':[{k:v for k,v in r.items() if k!='original_receipt'} for r in rows],
        'required_methods_missing':[list(x) for x in sorted(required-present)],'all_six_required_methods_present':required<=present,
        'zip_path':str(target),'zip_sha256':sha256_file(target),'private_score':None,
        'note':'Submit all six distinct required methods separately; the winner copy is not a new method or an independent test estimate.'}
    atomic_json(output/'comparison_report.json',report)
    with (output/'comparison.csv').open('w',newline='',encoding='utf-8') as stream:
        columns=['experiment_id','family','method','dev_macro_f1','micro_f1','subset_accuracy','trainable_parameters']
        writer=csv.DictWriter(stream,columns,extrasaction='ignore');writer.writeheader();writer.writerows(report['runs'])
    return report
''')


[Markdown Cell]

### config.py

Typed immutable learning choices, two-GPU constraints and exact recovery identities. Label count is inferred from the checksummed public taxonomy.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('config.py', r'''# CELL: Explicit immutable learning choices and portable input paths
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class LSIConfig:
    """Whole-case classification; one weighted target per original case."""
    experiment_id: str = 'P5_01_A_Full'
    family: str = 'a'
    method: str = 'full'
    run_dir: str = '/kaggle/working/cs781_phase5/P5_01_A_Full'
    input_root: str = '/kaggle/input'
    prepared_dir: str | None = None
    parent_run_dir: str | None = None
    tokenizer_path: str | None = None
    metadata_path: str | None = None
    resume_run_dir: str | None = None
    teacher_run_dir: str | None = None
    label_count: int = 0
    prompt_tokens: int = 16
    loss_type: str = 'bce'
    negative_gamma: float = 2.0
    positive_weight_cap: float = 16.0
    export_teacher_logits: bool = False
    expected_parent_sha256: str | None = None
    parent_checkpoint: str = 'best'
    max_epochs: int = 4
    minimum_epochs: int = 2
    patience: int = 2
    target_total_epochs: int | None = None
    seed: int = 781
    world_size: int = 2
    micro_batch_size: int = 1
    accumulation_steps: int = 4
    num_workers: int = 0
    max_run_seconds: int = 180 * 60
    min_free_bytes: int = 1 << 30
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    chunk_tokens: int = 256
    chunk_micro_batch: int = 8
    prefetch_batches: bool = True
    log_every_steps: int = 10
    inference_cases_per_batch: int = 4
    inference_chunk_budget: int = 64
    frozen_prefix_cache_tokens: int = 131072
    head_width: int = 128
    dropout: float = 0.2
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('config.py', r'''    pooling: str = 'attention'
    representation: str = 'all'
    section_chunks: int = 4
    recurrent_chunks: bool = True
    checkpoint_encoder: bool = True
    backbone_lr: float = 2e-5
    head_lr: float = 2e-4
    weight_decay: float = 0.01
    warmup_fraction: float = 0.1
    min_lr_ratio: float = 0.1
    max_grad_norm: float = 1.0
    class_weights: str = 'auto'
    rank: int = 16
    alpha: float = 16.0
    adapter_width: int = 32
    partial_layers: int = 2
    distillation_temperature: float = 2.0
    distillation_weight: float = 0.25
    calibrate_threshold: bool = True
    calibration_margin: float = 0.002

    def validate(self) -> None:
        matrix = {'full': 'a', 'lorma': 'a', 'prompt': 'b', 'adapters': 'c', 'partial': 'd', 'kd': 'b'}
        if matrix.get(self.method) != self.family:
            raise ValueError('Use the five distinct Phase 5 methods across A/B/C/D plus B distillation')
        if not self.experiment_id or any(c in self.experiment_id for c in '/\\'):
            raise ValueError('Use a simple experiment ID without path separators')
        if self.label_count < 0 or not 1 <= self.prompt_tokens <= 64:
            raise ValueError('Label dimension is inferred from prepared data; prompt length must be 1..64')
        if self.loss_type not in ('bce','asymmetric') or not 0 <= self.negative_gamma <= 4 or self.positive_weight_cap < 1:
            raise ValueError('Invalid multilabel loss configuration')
        if self.world_size != 2 or min(self.micro_batch_size, self.accumulation_steps) < 1:
            raise ValueError('Use two GPUs and positive accumulation/microbatch sizes')
        if self.num_workers != 0 or self.checkpoint_every_steps != 500:
            raise ValueError('Exact deterministic recovery uses zero loader workers and 500-update checkpoints')
        if min(self.max_epochs, self.minimum_epochs, self.patience) < 1:
            raise ValueError('Epoch counts and patience must be positive')
        if self.target_total_epochs is not None and self.target_total_epochs < 1:
            raise ValueError('Target total epochs must be positive')
        if not 16 <= self.chunk_tokens <= 512 or not 1 <= self.chunk_micro_batch <= 16:
            raise ValueError('Invalid chunk token/microbatch dimensions')
        if not 32 <= self.head_width <= 512 or self.head_width % 2 or not 0 <= self.dropout < 1:
            raise ValueError('Use an even head width and valid dropout')
        if self.pooling not in ('mean', 'attention') or self.representation not in ('all', 'first_last'):
            raise ValueError('Unknown pooling/representation policy')
        if self.section_chunks < 1 or self.partial_layers < 1:
            raise ValueError('Section/chosen-layer counts must be positive')
        if self.class_weights not in ('none', 'sqrt', 'auto'):
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('config.py', r'''            raise ValueError('Class weights must be none/sqrt/auto and use training labels only')
        if not 0 < self.backbone_lr <= 1e-3 or not 0 < self.head_lr <= 1e-2:
            raise ValueError('Invalid learning rates')
        if not 0 < self.distillation_weight < 1 or not 1 <= self.distillation_temperature <= 10:
            raise ValueError('Invalid distillation settings')
        if not 1 <= self.rank <= 128 or not 0 < self.alpha <= 256 or not 4 <= self.adapter_width <= 256:
            raise ValueError('Invalid low-rank/adapter dimensions')
        if self.max_run_seconds < 1 or not 0 <= self.warmup_fraction < 1 or not 0 < self.min_lr_ratio <= 1:
            raise ValueError('Invalid time/scheduler settings')
        if self.parent_checkpoint not in ('best', 'latest'):
            raise ValueError('Unknown parent checkpoint policy')
        if self.log_every_steps < 1 or self.inference_cases_per_batch < 1 or self.inference_chunk_budget < 1:
            raise ValueError('Invalid performance reporting/inference budget')
        if self.frozen_prefix_cache_tokens < 1:
            raise ValueError('Frozen-prefix cache token budget must be positive')

    def choices(self) -> dict:
        """Learning identity excludes transport paths and explicit duration requests."""
        value = asdict(self)
        for name in ('run_dir', 'input_root', 'prepared_dir', 'parent_run_dir', 'tokenizer_path', 'metadata_path',
                     'resume_run_dir', 'teacher_run_dir', 'expected_parent_sha256', 'parent_checkpoint',
                     'max_epochs', 'target_total_epochs', 'max_run_seconds', 'min_free_bytes'):
            value.pop(name)
        value.pop('export_teacher_logits')
        return value
''')


[Markdown Cell]

### data.py

Lossless uint16 memory maps retain every token and final document tail. Each real case supplies one multi-hot label vector; dummy tail slots are masked in the objective.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('data.py', r'''# CELL: Lossless uint16 case caches, no hidden document/chunk cap
from array import array
import math
from pathlib import Path
import re
import sys
import unicodedata
import numpy as np
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .io_utils import atomic_json, ensure_space, identity, read_json, records, sha256_file


def verify_cache(root: Path, expected: str) -> dict:
    value = read_json(root / 'case_cache_manifest.json')
    if value.get('version') != 'lsi-cache-v1' or value['identity'] != expected:
        raise ValueError('Wrong case-cache identity')
    if identity(value['choices']) != expected or identity({k:v for k,v in value.items() if k != 'manifest_sha256'}) != value.get('manifest_sha256'):
        raise ValueError('Changed case-cache metadata manifest')
    for name, receipt in value['artifacts'].items():
        if Path(name).name != name or sha256_file(root / name) != receipt['sha256'] or (root / name).stat().st_size != receipt['bytes']:
            raise ValueError('Changed/missing case cache: ' + name)
    return {**value, 'directory': str(root)}


def cache_data(prepared: Path, tokenizer_path: Path, cfg) -> dict:
    """Stream all cases once; small metadata stays in RAM and token IDs stay on disk."""
    if sys.byteorder != 'little':
        raise RuntimeError('Kaggle case caches use little-endian uint16')
    source = read_json(prepared / 'prepared_manifest.json')
    choices = {'prepared': source['identity'], 'tokenizer_sha256': sha256_file(tokenizer_path),
               'tokenizers': __import__('tokenizers').__version__, 'chunk_tokens': cfg.chunk_tokens}
    digest = identity(choices)
    root = Path(cfg.run_dir) / 'cache' / digest[:16]
    if (root / 'case_cache_manifest.json').is_file():
        return verify_cache(root, digest)
    # Reuse identical A/B caches from a saved profile/training run without copying bytes.
    for path in sorted(Path(cfg.input_root).rglob('case_cache_manifest.json')):
        if read_json(path).get('identity') == digest:
            return verify_cache(path.parent, digest)
    root.mkdir(parents=True, exist_ok=True)
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    if tokenizer.get_vocab_size() != 32768:
        raise ValueError('Use the matching own 32768-entry tokenizer')
    cases = {}
    token_count = 0
    counts = [0] * cfg.label_count
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('data.py', r'''    capacity = cfg.chunk_tokens - 2
    with (root / 'tokens.bin.partial').open('wb') as stream:
        for split in ('train', 'dev', 'test'):
            current = []
            for row in tqdm(records(prepared / (split + '.jsonl')), total=source['counts'][split]['cases'], desc='Tokenize ' + split):
                encoded = tokenizer.encode(row['text'], add_special_tokens=False).ids
                if any(x < 0 or x >= 32768 for x in encoded):
                    raise ValueError('Tokenizer produced an invalid embedding ID')
                ensure_space(root, len(encoded) * 2, cfg.min_free_bytes)
                array('H', encoded).tofile(stream)
                normalized = re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', row['text'])).strip()
                current.append({'id': row['id'], 'begin': token_count, 'length': len(encoded),
                                'chunks': max(1, math.ceil(len(encoded) / capacity)),
                                'gold': row.get('gold', []), 'source_id':row.get('source_id',row['id']),
                                'group': identity(normalized)})
                token_count += len(encoded)
                if split == 'train':
                    for label in row['gold']:
                        counts[label] += 1
            # Keep duplicate text and repeated source-ID views in the same
            # conditional calibration fold, including transitive connections.
            parent=list(range(len(current)))
            def find(i):
                while parent[i]!=i:
                    parent[i]=parent[parent[i]];i=parent[i]
                return i
            links={}
            for i,record in enumerate(current):
                for key in (('source',record['source_id']),('text',record['group'])):
                    if key in links:parent[find(i)]=find(links[key])
                    links[key]=i
            for i,record in enumerate(current):record['group']=identity([split,find(i)])
            cases[split] = current
    (root / 'tokens.bin.partial').replace(root / 'tokens.bin')
    value = {'version': 'lsi-cache-v1', 'identity': digest, 'choices': choices, 'cases': cases,
             'token_count': token_count, 'class_counts': counts, 'cases_dropped': 0,
             'all_document_tails_preserved': True,
             'artifacts': {'tokens.bin': {'sha256': sha256_file(root / 'tokens.bin'),
                                         'bytes': (root / 'tokens.bin').stat().st_size}}}
    value['manifest_sha256'] = identity(value)
    atomic_json(root / 'case_cache_manifest.json', value)
    return {**value, 'directory': str(root)}


def training_weights(cache: dict, cfg) -> np.ndarray:
    counts = np.asarray(cache['class_counts'], dtype=np.float64)
    n = len(cache['cases']['train'])
    ratio = np.maximum(1., (n-counts) / np.maximum(1.,counts))
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('data.py', r'''    weights = np.clip(np.sqrt(ratio), 1., cfg.positive_weight_cap)
    return weights.astype(np.float32) if cfg.class_weights != 'none' else np.ones_like(counts,dtype=np.float32)


class CaseDataset:
    """One original case -> all chunks; labels remain separate from model inputs."""
    def __init__(self, cache: dict, split: str, cfg, teacher=None, representation=None):
        self.cache, self.split, self.cfg, self.teacher = cache, split, cfg, teacher
        self.policy = representation or cfg.representation
        self.rows = cache['cases'][split]
        self.tokens = np.memmap(Path(cache['directory']) / 'tokens.bin', dtype='<u2', mode='r') if cache['token_count'] else np.empty(0, dtype=np.uint16)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        if index < 0:
            return {'chunks': [[1, 2]], 'positions': [[0., 0., 0.]], 'content_weights': [1.],
                    'gold': np.full(self.cfg.label_count,-100,dtype=np.float32), 'index': -1,
                    'teacher': np.zeros(self.cfg.label_count, dtype=np.float32),
                    'kept_tokens': 0, 'total_tokens': 0}
        row = self.rows[index]
        capacity = self.cfg.chunk_tokens - 2
        indices = list(range(row['chunks']))
        if self.policy == 'first_last':
            count = self.cfg.section_chunks
            indices = sorted(set(indices[:count] + indices[-count:]))
        chunks, positions, weights = [], [], []
        for slot in indices:
            first = slot * capacity
            length = min(capacity, max(0, row['length'] - first))
            ids = self.tokens[row['begin'] + first:row['begin'] + first + length].astype(np.int64).tolist()
            chunks.append([1, *ids, 2])
            positions.append([slot / max(1, row['chunks'] - 1), math.log1p(row['chunks']) / 10, length / capacity])
            weights.append(float(max(1, length)))
        gold=np.zeros(self.cfg.label_count,dtype=np.float32)
        gold[row['gold']]=1.
        return {'chunks': chunks, 'positions': positions, 'content_weights': weights,
                'gold': gold, 'index': index,
                'teacher': np.asarray(self.teacher[index] if self.teacher is not None else np.zeros(self.cfg.label_count), dtype=np.float32),
                'kept_tokens': sum(len(x) - 2 for x in chunks), 'total_tokens': row['length']}

    def close(self):
        if getattr(self.tokens, '_mmap', None) is not None:
            self.tokens._mmap.close()


# CELL: Ragged token chunks and padded document sequences
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('data.py', r'''def collate(rows: list[dict]) -> dict[str, torch.Tensor]:
    chunks = [chunk for row in rows for chunk in row['chunks']]
    width = max(map(len, chunks))
    count = max(len(r['chunks']) for r in rows)
    ids = torch.zeros(len(chunks), width, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    positions = torch.zeros(len(rows), count, 3)
    chunk_mask = torch.zeros(len(rows), count, dtype=torch.bool)
    content_weights = torch.zeros(len(rows), count)
    mapping = []
    first = 0
    for batch, row in enumerate(rows):
        length = len(row['chunks'])
        chunk_mask[batch, :length] = True
        positions[batch, :length] = torch.tensor(row['positions'])
        content_weights[batch, :length] = torch.tensor(row['content_weights'])
        for slot, chunk in enumerate(row['chunks']):
            ids[first, :len(chunk)] = torch.tensor(chunk)
            mask[first, :len(chunk)] = True
            mapping.append(batch * count + slot)
            first += 1
    return {'input_ids': ids, 'attention_mask': mask, 'chunk_document': torch.tensor(mapping),
            'chunk_mask': chunk_mask, 'positions': positions, 'content_weights': content_weights,
            'targets': torch.from_numpy(np.stack([r['gold'] for r in rows]).astype(np.float32)),
            'indices': torch.tensor([r['index'] for r in rows]),
            'case_lengths': torch.tensor([len(r['chunks']) for r in rows], dtype=torch.long),
            'token_lengths': torch.tensor([len(chunk) for chunk in chunks], dtype=torch.long),
            'teacher': torch.from_numpy(np.stack([r['teacher'] for r in rows]))}


class CasePlan:
    """Length-bucketed shuffles; every case is owned once, with masked dummy tails."""
    def __init__(self, rows, cfg, rank, epoch):
        self.cfg, self.rank = cfg, rank
        self.global_size = cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps
        self.steps = math.ceil(len(rows) / self.global_size)
        generator = np.random.default_rng(cfg.seed + epoch)
        # Shuffle ties, then group similarly long cases into globally balanced updates.
        tie_order = generator.permutation(len(rows))
        ordered = sorted(tie_order, key=lambda i: rows[i]['chunks'])
        blocks = [ordered[i:i + self.global_size] for i in range(0, len(rows), self.global_size)]
        blocks = [blocks[i] for i in generator.permutation(len(blocks))]
        self.indices = np.concatenate([np.pad(generator.permutation(block), (0, self.global_size - len(block)), constant_values=-1) for block in blocks])

    def global_indices(self, step):
        return self.indices[step * self.global_size:(step + 1) * self.global_size]

    def micro_indices(self, step, micro):
''')


[Markdown Cell]

Importable source part 5

In [ ]:
# [Code Cell]
append_module('data.py', r'''        return self.global_indices(step).reshape(self.cfg.accumulation_steps, self.cfg.world_size, self.cfg.micro_batch_size)[micro, self.rank].tolist()
''')


[Markdown Cell]

### io_utils.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('io_utils.py', r'''# CELL: Strict JSON, checksums and atomic writes
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shutil
import uuid

def strict_pairs(pairs: list[tuple[str, object]]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key}')
        result[key] = value
    return result

def reject_constant(value: str):
    raise ValueError(f'Nonfinite JSON value: {value}')

def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'), object_pairs_hook=strict_pairs,
                      parse_constant=reject_constant)

def records(path: str | Path):
    with Path(path).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                yield json.loads(line, object_pairs_hook=strict_pairs, parse_constant=reject_constant)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value: dict) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()

def atomic_json(path: str | Path, value: dict) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('io_utils.py', r'''
def ensure_space(directory: Path, needed: int, reserve: int) -> None:
    if shutil.disk_usage(directory).free < needed + reserve:
        raise OSError('Insufficient working disk. Start a fresh Kaggle session and attach the saved output; do not delete committed checkpoints.')

# CELL: Exclusive ownership and verified checkpoint copies
@contextmanager
def run_lock(directory: Path):
    """Linux advisory lock released by the OS after a killed Kaggle process."""
    import fcntl
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.training.lock').open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('This experiment is already running. Do not launch a second training cell.') from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def publish_copy(source: Path, destination: Path) -> None:
    """Publish a source without modifying it; hard-link only when supported."""
    if source.resolve() == destination.resolve():
        return
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        os.link(source, temporary)
    except OSError:
        shutil.copyfile(source, temporary)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
    os.replace(temporary, destination)
''')


[Markdown Cell]

### metrics.py

Match the organizer exactly: dense IDs zero through the largest reference label, including empty columns. Independent sigmoids, no top-k forcing. Conditional grouped calibration shrinks unstable rare-label thresholds to a shared threshold.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('metrics.py', r'''# CELL: Multilabel metrics with the organizer's dense label-axis convention
from pathlib import Path
import subprocess
import sys
import numpy as np
from sklearn.model_selection import GroupKFold
from .io_utils import atomic_json, read_json, records


def target_matrix(gold: list, dimension: int) -> np.ndarray:
    matrix=np.zeros((len(gold),dimension),dtype=np.uint8)
    for i,labels in enumerate(gold):
        if any(type(label) is not int or not 0<=label<dimension for label in labels):
            raise ValueError('Gold labels outside the public label axis')
        matrix[i,labels]=1
    return matrix


def metrics(gold: list, predicted: np.ndarray) -> dict:
    predicted=np.asarray(predicted,dtype=np.uint8)
    dimension=max((label for labels in gold for label in labels),default=-1)+1
    if not dimension or predicted.ndim!=2 or predicted.shape[0]!=len(gold) or predicted.shape[1]<dimension:
        raise ValueError('Nonempty multilabel truth and [N,C] decisions are required')
    actual=target_matrix(gold,dimension);predicted=predicted[:,:dimension]
    tp=(actual*predicted).sum(0,dtype=np.float64)
    support=actual.sum(0,dtype=np.float64);guesses=predicted.sum(0,dtype=np.float64)
    denominator=support+guesses
    f1=np.divide(2*tp,denominator,out=np.zeros_like(tp),where=denominator>0)
    return {'macro_f1':float(f1.mean()),'micro_f1':float(2*tp.sum()/max(1.,denominator.sum())),
        'subset_accuracy':float(np.all(actual==predicted,axis=1).mean()),'cases':len(gold),
        'evaluator_label_dimension':dimension,'per_label_f1':f1.tolist(),'per_label_support':support.astype(int).tolist(),
        'predicted_label_cardinality':float(predicted.sum(1).mean()),
        'scope':'Dense IDs 0..max(reference); absent columns count as zero F1, matching the supplied evaluator'}


def probabilities(logits: np.ndarray) -> np.ndarray:
    values=np.asarray(logits,dtype=np.float64)
    if values.ndim!=2 or not np.isfinite(values).all():raise ValueError('Logits must be finite [N,C] arrays')
    return np.exp(-np.logaddexp(0.,-values))


def decisions(probability, thresholds=.5):
    p=np.asarray(probability);thresholds=np.asarray(thresholds)
    if p.ndim!=2 or not np.isfinite(p).all() or not np.isfinite(thresholds).all():raise ValueError('Invalid probability or threshold array')
    return (p>=thresholds).astype(np.uint8)


def validate_prediction(predictions, rows, dimension=None):
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('metrics.py', r'''    if not isinstance(predictions,list) or len(predictions)!=len(rows):raise ValueError('Exactly one record per original test ID is required')
    ids=[]
    for record in predictions:
        if not isinstance(record,dict) or set(record)!={'id','pred'} or not isinstance(record['id'],str) or not isinstance(record['pred'],list):
            raise ValueError('Each record must have exactly string id and list pred')
        labels=record['pred']
        if any(type(x) is not int or x<0 or (dimension is not None and x>=dimension) for x in labels) or len(set(labels))!=len(labels):
            raise ValueError('Predictions require unique valid integer statute IDs')
        ids.append(record['id'])
    if ids!=[r['id'] for r in rows] or len(set(ids))!=len(ids):raise ValueError('Missing/extra/duplicate/reordered original case IDs')


def prediction(probability, rows, thresholds=.5, dimension=None):
    chosen=decisions(probability,thresholds)
    if chosen.shape[0]!=len(rows):raise ValueError('Logit and original ID counts differ')
    if dimension is not None:chosen=chosen[:,:dimension]
    result=[{'id':r['id'],'pred':np.flatnonzero(labels).astype(int).tolist()} for r,labels in zip(rows,chosen)]
    validate_prediction(result,rows,chosen.shape[1])
    return result


def official(prepared, prediction_path, expected, output):
    truth=[{'id':r['id'],'gold':r['gold']} for r in records(Path(prepared)/'dev.jsonl')]
    atomic_json(Path(output)/'dev_truth.json',truth)
    result=subprocess.run([sys.executable,str(Path(prepared)/'evaluate_phase5.py'),'--truth',str(Path(output)/'dev_truth.json'),
        '--prediction',str(prediction_path)],capture_output=True,text=True,check=True)
    line=next(x for x in result.stdout.splitlines() if x.startswith('Macro-F1:'))
    if abs(float(line.split(':')[1])-expected)>1e-6:raise ValueError('Native metric differs from official Phase 5 evaluator')
    (Path(output)/'official_dev_evaluation.txt').write_text(result.stdout,encoding='utf-8')


# CELL: Conditional grouped calibration; rare labels shrink to a shared threshold
def choose_threshold(probability, rows, enabled=True, margin=.002, seed=781):
    p=np.asarray(probability,dtype=np.float64);gold=[r['gold'] for r in rows]
    axis=p.shape[1];dimension=max(x for labels in gold for x in labels)+1
    y=target_matrix(gold,axis)
    grid=np.asarray([.001,.002,.005,.01,.02,.03,.05,.075,.1,.15,.2,.25,.3,.4,.5,.6,.7,.8])
    def fit(indices,per_label=False):
        # A fixed axis is essential: a fold missing the highest rare label must
        # still count that column, rather than changing the Macro-F1 denominator.
        truth=y[indices,:dimension]
        def score(t):
            pred=p[indices,:dimension]>=t
            tp=(truth*pred).sum(0);den=truth.sum(0)+pred.sum(0)
            return float(np.divide(2*tp,den,out=np.zeros(dimension),where=den>0).mean())
        scores=[score(t) for t in grid]
        best=max(range(len(grid)),key=lambda i:(scores[i],-abs(grid[i]-.5)))
        base=float(grid[best]);threshold=np.full(axis,base)
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('metrics.py', r'''        if per_label:
            for label in range(dimension):
                positive=int(y[indices,label].sum());negative=len(indices)-positive
                if min(positive,negative)<5:continue
                predicted=p[indices,label,None]>=grid[None]
                tp=(predicted*y[indices,label,None]).sum(0)
                f1=2*tp/np.maximum(1,predicted.sum(0)+positive)
                j=max(range(len(grid)),key=lambda k:(f1[k],-abs(grid[k]-base)))
                strength=positive/(positive+10.)
                logit=lambda x:np.log(x/(1-x))
                value=(1-strength)*logit(base)+strength*logit(grid[j])
                threshold[label]=1/(1+np.exp(-value))
        return threshold
    n=len(rows);full=np.arange(n);raw=metrics(gold,decisions(p,.5))['macro_f1']
    groups=np.asarray([r.get('group',r['id']) for r in rows]);unique=len(set(groups))
    candidates={'raw':np.full(axis,.5),'global':fit(full),'shrunk_per_label':fit(full,True)}
    fitted={name:metrics(gold,decisions(p,t))['macro_f1'] for name,t in candidates.items()}
    conditional={'raw':raw};folds=min(3,unique)
    if enabled and folds>=2:
        for name,per_label in (('global',False),('shrunk_per_label',True)):
            out=np.zeros_like(y)
            for train,held in GroupKFold(folds).split(full,groups=groups):
                threshold=fit(train,per_label)
                out[held]=decisions(p[held],threshold)
            conditional[name]=metrics(gold,out)['macro_f1']
    selected='raw'
    if enabled:
        eligible=[name for name in ('global','shrunk_per_label') if conditional.get(name,-1)>=raw+margin and fitted[name]>=raw+margin]
        if eligible:selected=max(eligible,key=lambda name:(conditional[name],fitted[name],name=='global'))
    return {'thresholds':candidates[selected].tolist(),'method':selected,'raw_dev_macro_f1':raw,
        'selected_dev_macro_f1':fitted[selected],'fitted_scores':fitted,'conditional_OOF_scores':conditional,
        'minimum_positives_and_negatives_for_label_threshold':5,'shrinkage_pseudocount':10,
        'folds':folds,'scope':'Conditional on the checkpoint already selected on this dev set; not an independent test estimate',
        'forced_top_k':False,'test_labels_used':False}


def case_diagnostics(gold, predicted, rows):
    result=metrics(gold,predicted)
    dimension=result['evaluator_label_dimension'];actual=target_matrix(gold,dimension)
    support=np.asarray(result['per_label_support']);f1=np.asarray(result['per_label_f1'])
    result['frequency_buckets']={}
    for lo,hi in ((1,5),(5,20),(20,100),(100,float('inf'))):
        keep=(support>=lo)&(support<hi)
        if keep.any():result['frequency_buckets'][f'{lo}:{hi}']={'labels':int(keep.sum()),'macro_f1':float(f1[keep].mean())}
    result['errors']=[{'id':r['id'],'gold':g,'pred':np.flatnonzero(p[:dimension]).tolist(),'tokens':r.get('length')}
        for r,g,p,a in zip(rows,gold,predicted,actual) if not np.array_equal(p[:dimension],a)][:30]
    return result
''')


[Markdown Cell]

### model.py

All chunks produce mean/last token features. A small contextual chunk GRU and shared attention/mean pooling are joined by statute-specific chunk attention. Independent outputs allow multiple laws. Prompt tokens never displace source content; GRU lengths remain on CPU across DDP.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('model.py', r'''# CELL: Token features -> bounded chunk projections -> one hierarchical case prediction
from dataclasses import replace
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.checkpoint import checkpoint
from .adaptation import adapt
from .parents import parent_class


class CaseClassifier(nn.Module):
    """All token chunks [P,T] -> contextual document chunks [B,K,H] -> [B,L].

    All full-document chunks are represented. Chunk microbatching and recomputation
    bound encoder activation memory without treating chunks as independent cases.
    """
    def __init__(self, backbone: nn.Module, width: int, cfg):
        super().__init__()
        self.backbone, self.cfg = backbone, cfg
        self.projection = nn.Sequential(nn.Linear(width * 2, cfg.head_width), nn.GELU(), nn.LayerNorm(cfg.head_width))
        self.positions = nn.Linear(3, cfg.head_width, bias=False)
        self.context = nn.GRU(cfg.head_width, cfg.head_width // 2, batch_first=True, bidirectional=True) if cfg.recurrent_chunks else None
        self.norm = nn.LayerNorm(cfg.head_width)
        self.attention = nn.Sequential(nn.Linear(cfg.head_width, cfg.head_width // 2), nn.Tanh(), nn.Linear(cfg.head_width // 2, 1)) if cfg.pooling == 'attention' else None
        self.dropout = nn.Dropout(cfg.dropout)
        if cfg.label_count < 1:
            raise ValueError('Prepared LSI label dimension must be known before model construction')
        self.classifier = nn.Linear(cfg.head_width * 2, cfg.label_count)
        self.label_queries = nn.Parameter(torch.empty(cfg.label_count, cfg.head_width))
        self.label_weights = nn.Parameter(torch.empty(cfg.label_count, cfg.head_width))
        nn.init.normal_(self.label_queries, std=.02)
        nn.init.normal_(self.label_weights, std=.02)

    def hidden_states(self, ids: torch.Tensor, mask: torch.Tensor, full: bool) -> torch.Tensor:
        """Same own feature path; padding is verified on CPU, avoiding GPU bool sync."""
        if self.cfg.family == 'b':
            if self.cfg.method == 'prompt':
                prefix = self.backbone.soft_prompt[None].expand(ids.shape[0], -1, -1)
                embedded = torch.cat((prefix, self.backbone.shared_embedding(ids)), dim=1)
                positions = torch.arange(embedded.shape[1], device=ids.device)
                if embedded.shape[1] > self.backbone.source_positions.num_embeddings:
                    raise ValueError('Prompt plus full content exceeds B source position capacity')
                hidden = self.backbone.dropout(embedded + self.backbone.source_positions(positions)[None])
                valid = torch.cat((torch.ones(ids.shape[0], prefix.shape[1], device=ids.device, dtype=torch.bool), mask), dim=1)
                for layer in self.backbone.encoder:
                    hidden = layer(hidden, valid[:, None, None, :])
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('model.py', r'''                # Pool original content only; prompt is context, never a new text token.
                return self.backbone.encoder_norm(hidden)[:, prefix.shape[1]:]
            return self.backbone.encode(ids, mask)
        if full:
            return self.backbone.hidden_states(ids, None)
        if self.cfg.family == 'd':
            hidden = self.backbone.token_embedding(ids)
            for layer in self.backbone.layers:
                hidden = layer(hidden, mask)
        else:
            causal = torch.ones((ids.shape[1], ids.shape[1]), device=ids.device, dtype=torch.bool).tril()
            allowed = causal[None, None] & mask[:, None, None, :]
            hidden = self.backbone.token_embedding(ids)
            modules = self.backbone.stages if self.cfg.family == 'c' else self.backbone.layers
            for layer in modules:
                hidden = layer(hidden, allowed)
        return self.backbone.final_norm(hidden)

    @staticmethod
    def pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Token features [p,T,D] -> FP32 mean/last [p,2D], excluding padding."""
        hidden = hidden.float()
        lengths = mask.sum(1).clamp_min(1)
        mean = (hidden * mask[..., None]).sum(1) / lengths[:, None]
        last = hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths - 1]
        return torch.cat((mean, last), dim=-1)

    def encoder_pool(self, ids, mask, full):
        return self.pool(self.hidden_states(ids, mask, full), mask)

    def suffix_pool(self, hidden, mask):
        for layer in self.backbone.layers[-self.cfg.partial_layers:]:
            hidden = layer(hidden, mask)
        return self.pool(self.backbone.final_norm(hidden), mask)

    def chunk_features(self, ids: torch.Tensor, mask: torch.Tensor, full=False, cache_prefix=True) -> torch.Tensor:
        """Checkpoint active encoder work; reuse D's prefix within its memory budget."""
        active_checkpoint = self.training and self.cfg.checkpoint_encoder
        if self.cfg.family == 'd' and self.cfg.method == 'partial' and cache_prefix:
            hidden = self.backbone.token_embedding(ids)
            valid = None if full else mask
            # Every prefix parameter is frozen and IDs are integers: no autograd
            # graph is created. Keep training mode/dropout and sparse route cache.
            for layer in self.backbone.layers[:-self.cfg.partial_layers]:
                hidden = layer(hidden, valid)
            assert not hidden.requires_grad, 'Frozen D prefix unexpectedly became trainable'
            # Pool mask still excludes right padding; attention may use None only
            # when every token is valid. No detach is applied to trainable suffixes.
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('model.py', r'''            if active_checkpoint:
                pooled = checkpoint(self.suffix_pool, hidden, mask, use_reentrant=False)
            else:
                pooled = self.suffix_pool(hidden, mask)
        elif active_checkpoint:
            pooled = checkpoint(self.encoder_pool, ids, mask, full, use_reentrant=False)
        else:
            pooled = self.encoder_pool(ids, mask, full)
        return self.projection(pooled)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                chunk_document: torch.Tensor, chunk_mask: torch.Tensor,
                positions: torch.Tensor, content_weights: torch.Tensor,
                return_attention: bool = False, case_lengths=None, pack_layout=None, cache_prefix=True):
        features = []
        if pack_layout is None:
            lengths = attention_mask.sum(1).cpu().tolist()
            pack_layout = make_pack_layout(lengths, self.cfg.chunk_micro_batch)
        for first, stop, width, full in pack_layout:
            features.append(self.chunk_features(input_ids[first:stop, :width], attention_mask[first:stop, :width], full, cache_prefix))
        batch, count = chunk_mask.shape
        flat = torch.zeros(batch * count, self.cfg.head_width, device=input_ids.device, dtype=torch.float32)
        flat = flat.index_copy(0, chunk_document, torch.cat(features).float())
        hidden = flat.reshape(batch, count, -1) + self.positions(positions)
        if self.context is not None:
            # DDP moves tensor arguments to its rank's CUDA device, including
            # metadata. Always construct this tensor on CPU for the packed GRU.
            if case_lengths is None:
                lengths = chunk_mask.sum(1).to(device='cpu', dtype=torch.long)
            elif isinstance(case_lengths, torch.Tensor):
                lengths = case_lengths.to(device='cpu', dtype=torch.long)
            else:
                lengths = torch.tensor(case_lengths, device='cpu', dtype=torch.long)
            packed = pack_padded_sequence(hidden, lengths, batch_first=True, enforce_sorted=False)
            packed, _ = self.context(packed)
            context, _ = pad_packed_sequence(packed, batch_first=True, total_length=count)
            hidden = hidden + context
        hidden = self.norm(hidden)
        mean_weights = content_weights * chunk_mask
        mean_weights = mean_weights / mean_weights.sum(1, keepdim=True).clamp_min(1)
        if self.attention is None or self.cfg.pooling == 'mean':
            weights = mean_weights
        else:
            scores = self.attention(hidden).squeeze(-1).float().masked_fill(~chunk_mask, float('-inf'))
            weights = torch.softmax(scores, dim=1)
        pooled = (hidden * weights[..., None]).sum(1)
        mean = (hidden * mean_weights[..., None]).sum(1)
        label_scores = torch.einsum('bch,lh->blc', hidden.float(), self.label_queries.float()) / self.cfg.head_width ** .5
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('model.py', r'''        label_scores = label_scores.masked_fill(~chunk_mask[:, None], float('-inf'))
        label_attention = torch.softmax(label_scores, dim=-1)
        label_features = torch.einsum('blc,bch->blh', label_attention, hidden.float())
        logits = self.classifier(self.dropout(torch.cat((pooled, mean), dim=-1)))
        logits = logits + torch.einsum('blh,lh->bl', self.dropout(label_features), self.label_weights.float())
        return (logits, weights) if return_attention else logits


def forward(model, batch: dict, device: torch.device, return_attention=False):
    names = ('input_ids', 'attention_mask', 'chunk_document', 'chunk_mask', 'positions', 'content_weights')
    actual = model.module if hasattr(model, 'module') else model
    lengths = (batch['token_lengths'] if 'token_lengths' in batch else batch['attention_mask'].sum(1)).tolist()
    layout = make_pack_layout(lengths, actual.cfg.chunk_micro_batch)
    counts = batch['case_lengths'] if 'case_lengths' in batch else batch['chunk_mask'].sum(1)
    # Python scalars survive DDP's recursive CUDA transfer unchanged. This also
    # avoids moving CPU lengths to CUDA and synchronizing them back on every call.
    case_lengths = tuple(int(value) for value in counts.tolist())
    return model(**{name: batch[name].to(device, non_blocking=True) for name in names},
                 return_attention=return_attention, case_lengths=case_lengths, pack_layout=layout,
                 cache_prefix=sum(lengths) <= actual.cfg.frozen_prefix_cache_tokens)


def make_pack_layout(lengths: list[int], micro_batch: int) -> tuple:
    """CPU-only contiguous packs: trim padding, preserve token and chunk order."""
    result = []
    for first in range(0, len(lengths), micro_batch):
        stop = min(len(lengths), first + micro_batch)
        current = lengths[first:stop]
        width = max(current)
        result.append((first, stop, width, min(current) == width))
    return tuple(result)


def build_model(job: dict, cfg, device: torch.device, load_parent=True) -> CaseClassifier:
    Config, Model = parent_class(job['parent_source_dir'], cfg.family, job['parent_contract'])
    changes = {'activation_checkpointing': False}
    if cfg.family == 'd':
        changes['sparse_attention_checkpointing'] = False
    base_cfg = replace(Config(**job['parent']['config']), **changes)
    with torch.device('meta'):
        backbone = Model(base_cfg)
    if load_parent:
        path = Path(job['parent']['checkpoint'])
        if path.stat().st_size != job['parent']['checkpoint_bytes'] or path.stat().st_mtime_ns != job['parent']['checkpoint_mtime_ns']:
            raise ValueError('Verified foundation checkpoint changed before loading')
        payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        backbone.load_state_dict(payload['model'], strict=True, assign=True)
        del payload
''')


[Markdown Cell]

Importable source part 5

In [ ]:
# [Code Cell]
append_module('model.py', r'''    else:
        backbone._apply(lambda tensor: torch.empty_like(tensor, device=device) if tensor.is_meta else tensor.to(device))
    foundation = sum(p.numel() for p in backbone.parameters())
    assert 50_000_000 < foundation < 300_000_000
    capacity = base_cfg.max_source_tokens if cfg.family == 'b' else base_cfg.max_seq_tokens
    if cfg.chunk_tokens + (cfg.prompt_tokens if cfg.method == 'prompt' else 0) > capacity:
        raise ValueError('chunk_tokens exceeds the actual foundation context capacity')
    changed = adapt(backbone, cfg)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(cfg.seed + 1000)
        model = CaseClassifier(backbone, base_cfg.d_model, cfg).to(device)
    if job.get('cache'):
        # Prior uses training frequencies only, never dev/test prevalence.
        counts = torch.tensor(job['cache']['class_counts'], device=device, dtype=torch.float32)
        prior = ((counts + .5) / (len(job['cache']['cases']['train']) + 1.)).clamp(.001,.999)
        with torch.no_grad():
            model.classifier.bias.copy_(torch.logit(prior))
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 50_000_000 < total < 300_000_000 and trainable > 0
    model.parameter_report = {'foundation_unique_parameters': foundation, 'total_unique_parameters': total,
        'trainable_parameters': trainable, 'active_backbone_trainable_parameters': sum(p.numel() for p in backbone.parameters() if p.requires_grad),
        'adapted_names': changed, 'inactive_B_decoder': cfg.family == 'b'}
    return model
''')


[Markdown Cell]

### parents.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('parents.py', r'''# CELL: Match a parent by source and data rather than a fixed epoch count
import hashlib
import importlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file

def source_hashes(directory: Path) -> dict[str,str]:
    return {path.name: sha256_file(path) for path in sorted(directory.glob('*.py'))}

def learning_choices(values: dict, contract: dict) -> dict:
    return {key:value for key,value in values.items() if key not in contract['fingerprint_exclusions']}

def reproduce_parent(values: dict, contract: dict, version: str) -> str:
    payload={'version':contract['runtime_version'],'config':learning_choices(values,contract),
             'data':contract['data_fingerprint'],'sources':contract['source_sha256'],'torch':version}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def intact_pairs(root: Path) -> tuple[list, list]:
    """Check receipts/checkpoint bytes incrementally before mmap loading."""
    pairs=[]; warnings=[]
    for name in ('slot_0.pt','slot_1.pt','best.pt'):
        path=root/'checkpoints'/name; receipt_path=path.with_name(name+'.json')
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt=read_json(receipt_path)
            if any(type(receipt.get(key)) is not int or receipt[key]<0 for key in ('bytes','epoch','step','generation')):
                raise ValueError('Invalid receipt counters')
            if receipt['bytes']!=path.stat().st_size or receipt['sha256']!=sha256_file(path):
                raise ValueError('Size/SHA-256 mismatch')
            pairs.append((path,receipt))
        except (ValueError,OSError,KeyError,TypeError) as error:
            warnings.append(f'{path}: {error}')
    if not pairs:
        raise FileNotFoundError('No intact full checkpoint/receipt pair. ' + '; '.join(warnings))
    return pairs,warnings

def parent_candidates(cfg, contract: dict) -> list[dict]:
    """Return useful rejection reasons for every discovered run root."""
    roots=[Path(cfg.parent_run_dir)] if cfg.parent_run_dir else sorted({p.parent for p in Path(cfg.input_root).rglob('train_config.json')})
    rows=[]
    for root in roots:
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('parents.py', r'''        row={'run_dir':str(root),'reasons':[],'accepted':False}
        try:
            required=('train_config.json','data_report.json','training_summary.json',f'code/model_{cfg.family}/model.py')
            missing=[name for name in required if not (root/name).is_file()]
            if missing:
                row['reasons'].append('Missing: '+', '.join(missing))
            else:
                saved=read_json(root/'train_config.json'); summary=read_json(root/'training_summary.json')
                row.update(planned_epochs=saved.get('epochs'),completed_epochs=summary.get('epoch'),step=summary.get('step'),status=summary.get('status'))
                if type(saved.get('epochs')) is not int or saved['epochs']<1:
                    row['reasons'].append('Invalid configured epoch count')
                actual=source_hashes(root/f'code/model_{cfg.family}')
                changed=sorted(name for name in set(actual)|set(contract['source_sha256']) if actual.get(name)!=contract['source_sha256'].get(name))
                if changed:
                    row['reasons'].append('Original source differs: '+', '.join(changed))
                if read_json(root/'data_report.json').get('data_fingerprint')!=contract['data_fingerprint']:
                    row['reasons'].append('Wrong preprocessing identity for this model')
                names=[name for name in ('slot_0.pt','slot_1.pt','best.pt') if (root/'checkpoints'/name).is_file()
                       and (root/'checkpoints'/(name+'.json')).is_file()]
                row['checkpoint_pairs']=names
                if not names:
                    row['reasons'].append('Attach saved output with real .pt files and .pt.json receipts; .ipynb/prediction ZIP is insufficient')
                row['accepted']=not row['reasons']
        except (ValueError,OSError,KeyError,TypeError,AttributeError) as error:
            row['reasons'].append(f'{type(error).__name__}: {error}')
        rows.append(row)
    return rows

# CELL: Check real cursor/config/Torch identity before selecting weights
def validate_payload(payload: dict, receipt: dict, values: dict, contract: dict) -> None:
    required={'model','config','cursor','optimizer','scaler','scheduler','rng_by_rank','world_size'}
    if required-payload.keys():
        raise ValueError('Parent checkpoint lacks full recovery state')
    cursor=payload['cursor']
    if (any(type(cursor.get(key)) is not int or cursor[key]<0 for key in ('epoch','next_batch','global_step','optimizer_steps'))
            or payload['fingerprint']!=receipt['fingerprint'] or payload['step']!=receipt['step']
            or payload['epoch']!=receipt['epoch'] or payload['epoch']!=cursor['epoch']
            or payload['step']!=cursor['global_step'] or cursor['epoch']>values['epochs']
            or cursor['optimizer_steps']>cursor['global_step']
            or payload['scheduler']['steps']!=cursor['optimizer_steps']
            or payload['world_size']!=values['world_size'] or len(payload['rng_by_rank'])!=values['world_size']
            or learning_choices(payload['config'],contract)!=learning_choices(values,contract)):
        raise ValueError('Parent cursor/config/scheduler/RNG/receipt disagree')

def verify_parent(cfg, contract: dict) -> tuple[Path,dict]:
    diagnostics=parent_candidates(cfg,contract)
    atomic_json(Path(cfg.run_dir)/'parent_input_report.json',{'candidates':diagnostics})
    print(json.dumps({'parent_candidates':diagnostics},indent=2),flush=True)
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('parents.py', r'''    accepted=[Path(row['run_dir']) for row in diagnostics if row['accepted']]
    if len(accepted)!=1:
        raise ValueError(f'Found {len(accepted)} matching Model {cfg.family.upper()} outputs. Attach ONE full saved '
                         'output or set parent_run_dir to its exact printed run_dir; see parent_input_report.json.')
    root=accepted[0]; values=read_json(root/'train_config.json')
    pairs,warnings=intact_pairs(root)
    latest=max(pairs,key=lambda item:(item[1]['step'],item[1]['generation']))
    payload=torch.load(latest[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(payload,latest[1],values,contract)
    versions=list(contract['recorded_torch_candidates'])+[str(torch.__version__)]
    for name in ('continuation_runtime_identity.json','training_runtime_identity.json'):
        path=root/name
        if path.is_file():
            record=read_json(path)
            if isinstance(record.get('torch'),str):
                versions.insert(0,record['torch'])
    matched=next((v for v in dict.fromkeys(versions) if reproduce_parent(values,contract,v)==payload['fingerprint']),None)
    if matched is None:
        raise ValueError('Cannot reproduce original config/source/data/Torch fingerprint. Attach the matching original runtime and tokenizer; do not disable this check.')
    cursor=payload['cursor']
    if cursor['epoch']!=values['epochs'] or cursor['next_batch']!=0:
        raise ValueError(f"Parent has {cursor['epoch']}/{values['epochs']} completed epochs, next_batch={cursor['next_batch']}. Finish/save its continuation before starting a new LSI experiment.")
    report=read_json(root/'data_report.json')
    examples=report.get('train_examples')
    coverage=(isinstance(report.get('eligible_source_tokens'),dict) if cfg.family=='b' else report.get('full_window_coverage') is True)
    if type(examples) is not int or examples<1 or not coverage:
        raise ValueError('Parent data report lacks a positive exhaustive training-window count')
    global_batch=values['world_size']*values['micro_batch_size']*values['accumulation_steps']
    expected_steps=math.ceil(examples/global_batch)*cursor['epoch']
    if cursor['global_step']!=expected_steps:
        raise ValueError(f"Parent counters disagree with its exhaustive window plan: expected {expected_steps} boundaries, got {cursor['global_step']}")
    summary=read_json(root/'training_summary.json')
    if summary.get('status')!='complete' or summary.get('epoch')!=cursor['epoch'] or summary.get('step')!=cursor['global_step'] or summary.get('fingerprint')!=payload['fingerprint']:
        raise ValueError('Parent training summary disagrees with its latest intact checkpoint')
    chosen=next((item for item in pairs if item[0].name=='best.pt'),latest) if cfg.parent_checkpoint=='best' else latest
    selected=torch.load(chosen[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(selected,chosen[1],values,contract)
    if selected['fingerprint']!=payload['fingerprint']:
        raise ValueError('Selected parent best/latest belongs to another run')
    for name,tensor in selected['model'].items():
        if tensor.is_floating_point():
            if tensor.dtype!=torch.float32 or not all(bool(torch.isfinite(c).all()) for c in tensor.reshape(-1).split(1_000_000)):
                raise ValueError(f'Nonfinite/non-FP32 master weights: {name}')
    result={'family':cfg.family,'config':values,'completed_epochs':cursor['epoch'],'planned_epochs':values['epochs'],
            'checkpoint':str(chosen[0]),'checkpoint_sha256':chosen[1]['sha256'],'checkpoint_bytes':chosen[1]['bytes'],
            'checkpoint_mtime_ns':chosen[0].stat().st_mtime_ns,'checkpoint_epoch':chosen[1]['epoch'],'checkpoint_step':chosen[1]['step'],
            'selection':cfg.parent_checkpoint,'original_fingerprint':payload['fingerprint'],'original_torch':matched,
            'data_fingerprint':contract['data_fingerprint'],'source_sha256':contract['source_sha256'],'checkpoint_warnings':warnings}
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('parents.py', r'''    print(f"Verified Model {cfg.family.upper()}: {cursor['epoch']} completed epochs; selected {chosen[0].name} at epoch {chosen[1]['epoch']}",flush=True)
    return root,result

# CELL: Find exactly matching small preprocessing artifacts without reading LM binaries
def tokenizer_inputs(cfg, contract: dict) -> tuple[Path,Path,dict]:
    metadata=[Path(cfg.metadata_path)] if cfg.metadata_path else sorted(Path(cfg.input_root).rglob('metadata.json'))
    matches=[]
    required=('train.bin','val.bin','tokenizer.json','train_documents.jsonl','validation_documents.jsonl','validation_pairs.jsonl')
    for path in metadata:
        if not path.is_file():
            continue
        try:
            value=read_json(path)
            hashes={name:value['artifacts'][name]['sha256'] for name in required}
            hashes['metadata.json']=sha256_file(path)
            fingerprint=hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
            if fingerprint==contract['data_fingerprint']:
                matches.append((path,value))
        except (KeyError,TypeError,ValueError,OSError):
            continue
    if not matches:
        raise FileNotFoundError(f'Missing matching Model {cfg.family.upper()} metadata.json. Use its original preprocessing output, not another token-budget export; large .bin files are unnecessary.')
    meta,value=matches[0]; expected=value['artifacts']['tokenizer.json']['sha256']
    candidates=[Path(cfg.tokenizer_path)] if cfg.tokenizer_path else [meta.parent/'tokenizer.json',*Path(cfg.input_root).rglob('tokenizer.json')]
    token=next((p for p in candidates if p.is_file() and sha256_file(p)==expected),None)
    if token is None:
        raise FileNotFoundError('No tokenizer.json matches the verified metadata; attach the original tokenizer')
    tokenizer=Tokenizer.from_file(str(token))
    specials={'<pad>':0,'<s>':1,'</s>':2,'<unk>':3,'<sep>':4,'<mask>':5}
    if tokenizer.get_vocab_size()!=32768 or any(tokenizer.token_to_id(k)!=v for k,v in specials.items()):
        raise ValueError('Unexpected parent tokenizer vocabulary/special IDs')
    return token,meta,{'tokenizer_sha256':expected,'metadata_sha256':sha256_file(meta)}

def parent_class(directory: str, family: str, contract: dict):
    """Import only the checksum-verified original config and model implementation."""
    root=Path(directory)
    if source_hashes(root)!=contract['source_sha256']:
        raise ValueError('Parent source changed after verification')
    package='verified_parent_'+family+'_'+contract['source_sha256']['model.py'][:12]
    if package not in sys.modules:
        spec=importlib.util.spec_from_file_location(package,root/'__init__.py',submodule_search_locations=[str(root)])
        module=importlib.util.module_from_spec(spec); sys.modules[package]=module; spec.loader.exec_module(module)
    config=importlib.import_module(package+'.config'); model=importlib.import_module(package+'.model')
    return config.TrainConfig,getattr(model,contract['class_name'])
''')


[Markdown Cell]

### performance.py

Bounded deterministic CPU prefetch overlaps collation/pinning with GPU work without changing saved row order or consuming training RNG.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('performance.py', r'''# CELL: Bounded deterministic CPU prefetch; no worker RNG or data sampling
from concurrent.futures import ThreadPoolExecutor
import torch
from .data import collate


class MicroBatches:
    """At most one CPU/pinned batch ahead of the exact saved row cursor.

    Collation uses no random transforms, device work or model state. Pending rows
    are discarded on a stop and regenerated from the same cursor on recovery.
    """
    def __init__(self, dataset, plan, first_step: int, cfg, pin_memory=True):
        self.dataset, self.cfg = dataset, cfg
        self.pin_memory = pin_memory and torch.cuda.is_available()
        self.indices = (plan.micro_indices(step, micro) for step in range(first_step, plan.steps)
                        for micro in range(cfg.accumulation_steps))
        self.executor = ThreadPoolExecutor(max_workers=1) if cfg.prefetch_batches else None
        self.pending = None

    def build(self, indices):
        batch = collate([self.dataset[index] for index in indices])
        if self.pin_memory:
            batch = {name: value.pin_memory() if name not in ('case_lengths', 'token_lengths', 'indices') else value
                     for name, value in batch.items()}
        return batch

    def __enter__(self):
        if self.executor:
            indices = next(self.indices, None)
            if indices is not None:
                self.pending = self.executor.submit(self.build, indices)
        return self

    def __next__(self):
        if self.executor is None:
            return self.build(next(self.indices))
        if self.pending is None:
            raise StopIteration
        batch = self.pending.result()
        indices = next(self.indices, None)
        self.pending = self.executor.submit(self.build, indices) if indices is not None else None
        return batch

    def __exit__(self, *_):
        if self.executor:
            if self.pending:
                self.pending.cancel()
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('performance.py', r'''            self.executor.shutdown(wait=True)
''')


[Markdown Cell]

### predict.py

Two GPU replicas process each original case once. Reload actual best tensors, report native dev metrics, freeze validation thresholds and only then infer test. Verify the single-root-file submission ZIP.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('predict.py', r'''# CELL: Full-case inference and strict learned-task reconstruction
import argparse
from dataclasses import replace
from pathlib import Path
import zipfile
import numpy as np
import torch
import torch.multiprocessing as mp
from tqdm.auto import tqdm
from .config import LSIConfig
from .data import CaseDataset, cache_data, collate, verify_cache
from .io_utils import atomic_json, read_json, sha256_file
from .metrics import case_diagnostics, choose_threshold, metrics, official, prediction, probabilities, validate_prediction
from .model import build_model, forward
from .parents import source_hashes
from .preparation import prepared_input


def logits_partition(model, job, cfg, split, device, rank=0, world_size=1, representation=None):
    dataset = CaseDataset(job['cache'], split, cfg, representation=representation)
    output = {}
    indices = balanced_indices(dataset.rows, rank, world_size)
    batches, current, work = [], [], 0
    for index in indices:
        count = dataset.rows[index]['chunks']
        if current and (len(current) >= cfg.inference_cases_per_batch or work + count > cfg.inference_chunk_budget):
            batches.append(current)
            current, work = [], 0
        current.append(index)
        work += count
    if current:
        batches.append(current)
    model.eval()
    progress = tqdm(total=len(indices), desc='Predict ' + split, disable=rank != 0, unit='cases')
    try:
        with torch.inference_mode():
            for selected in batches:
                batch = collate([dataset[index] for index in selected])
                with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    logits = forward(model, batch, device)
                values = logits.float().cpu().numpy()
                if values.shape != (len(selected), cfg.label_count) or not np.isfinite(values).all():
                    raise ValueError('Nonfinite/wrong-shaped case logits')
                output.update(zip(selected, values))
                progress.update(len(selected))
    finally:
        progress.close()
        dataset.close()
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('predict.py', r'''    return output


def balanced_indices(rows, rank, world_size):
    """Assign all cases once by estimated chunk work, then batch similar lengths."""
    assigned = [[] for _ in range(world_size)]
    work = [0] * world_size
    for index in sorted(range(len(rows)), key=lambda i: (-rows[i]['chunks'], i)):
        owner = min(range(world_size), key=lambda r: (work[r], r))
        assigned[owner].append(index)
        work[owner] += rows[index]['chunks']
    return sorted(assigned[rank], key=lambda i: (rows[i]['chunks'], i))


def inference_worker(rank, job_path, splits):
    """Independent GPU replicas for inference; training still uses NCCL DDP."""
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    job = read_json(job_path)
    root = Path(job['config']['run_dir'])
    model, local, cfg, _ = load_task(root, Path(job['prepared_dir']), torch.device('cuda', rank), root / 'inference_cache' / str(rank))
    directory = root / 'predictions'
    directory.mkdir(exist_ok=True)
    for split in splits:
        values = logits_partition(model, local, cfg, split, torch.device('cuda', rank), rank, 2)
        indices = sorted(values)
        logits = np.stack([values[i] for i in indices]) if indices else np.empty((0, cfg.label_count), dtype=np.float32)
        path = directory / (split + f'_rank{rank}.npz')
        with path.with_suffix('.npz.partial').open('wb') as stream:
            np.savez(stream, indices=np.asarray(indices, dtype=np.int64), logits=logits)
        path.with_suffix('.npz.partial').replace(path)
    atomic_json(directory / f'parameters_rank{rank}.json', model.parameter_report)


def parallel_predict(job_path, splits):
    """Merge exact original-row order only after both complete GPU shards exist."""
    job = read_json(job_path)
    directory = Path(job['config']['run_dir']) / 'predictions'
    mp.spawn(inference_worker, args=(str(job_path), tuple(splits)), nprocs=2, join=True)
    for split in splits:
        values = {}
        for rank in range(2):
            with np.load(directory / (split + f'_rank{rank}.npz'), allow_pickle=False) as shard:
                for index, logits in zip(shard['indices'], shard['logits']):
                    if int(index) in values:
                        raise ValueError('Inference ranks duplicated a case')
                    values[int(index)] = logits
        logits = ordered_logits(values, job['cache']['cases'][split])
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('predict.py', r'''        np.save(directory / (split + '_logits.npy'), logits)
        atomic_json(directory / (split + '_order.json'), [{'id': r['id']} for r in job['cache']['cases'][split]])
    return read_json(directory / 'parameters_rank0.json')


def ordered_logits(values, rows):
    if set(values) != set(range(len(rows))):
        raise ValueError('Inference must cover each original case exactly once')
    return np.stack([values[i] for i in range(len(rows))])


def score_dev(values, job):
    rows = job['cache']['cases']['dev']
    logits = ordered_logits(values, rows)
    cfg=LSIConfig(**job['config'])
    probability=probabilities(logits)
    calibration=choose_threshold(probability,rows,cfg.calibrate_threshold,cfg.calibration_margin,cfg.seed)
    predicted=(probability>=calibration['thresholds']).astype(int)
    result=case_diagnostics([r['gold'] for r in rows],predicted,rows)
    result['raw_dev_macro_f1']=calibration['raw_dev_macro_f1']
    result['calibration_method']=calibration['method']
    # Break equal-F1 plateaus by ranking quality, so rare-label probabilities
    # below 0.5 do not automatically lock best.pt to an untrained early epoch.
    from sklearn.metrics import average_precision_score
    from .metrics import target_matrix
    dimension=job['label_schema']['dev_evaluator_dimension']
    gold=target_matrix([r['gold'] for r in rows],dimension)
    result['macro_average_precision']=sum(average_precision_score(gold[:,j],probability[:,j]) if gold[:,j].any() else 0.
        for j in range(dimension))/dimension
    return result,prediction(probability,rows,calibration['thresholds'],dimension=dimension)


def load_task(root: Path, prepared: Path, device, workspace: Path):
    root, workspace = Path(root), Path(workspace)
    workspace.mkdir(parents=True, exist_ok=True)
    job = read_json(root / 'job.json')
    if job.get('version') != 'cs781-lsi-v1' or source_hashes(root / 'code/lsi') != job['runtime_source_sha256']:
        raise ValueError('Missing/changed LSI source; use the original complete saved output')
    if source_hashes(Path(__file__).parent) != job['runtime_source_sha256']:
        raise ValueError('Load this task with its exact saved runtime, not another notebook version')
    if job['environment']['torch'] != str(torch.__version__):
        raise ValueError('Use the saved inference PyTorch build ' + job['environment']['torch'])
    if read_json(prepared / 'prepared_manifest.json')['identity'] != job['prepared_identity']:
        raise ValueError('Prepared cases differ from the learned task')
    token = root / 'tokenizer/tokenizer.json'
    if sha256_file(token) != job['parent']['tokenizer_sha256']:
        raise ValueError('Task tokenizer changed')
    cfg = replace(LSIConfig(**job['config']), run_dir=str(workspace), prepared_dir=str(prepared))
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('predict.py', r'''    saved_cache = Path(job['cache']['directory'])
    cache = verify_cache(saved_cache, job['cache']['identity']) if (saved_cache / 'case_cache_manifest.json').is_file() else cache_data(prepared, token, cfg)
    if cache['manifest_sha256'] != job['cache']['manifest_sha256']:
        raise ValueError('Learned task case metadata changed')
    local = {**job, 'parent_source_dir': str(root / 'parent_source' / ('model_' + cfg.family)),
             'tokenizer_path': str(token), 'cache': cache}
    best = root / 'checkpoints/best.pt'
    receipt = read_json(root / 'checkpoints/best.pt.json')
    if sha256_file(best) != receipt['sha256'] or receipt['fingerprint'] != job['fingerprint']:
        raise ValueError('Learned task best checkpoint/receipt changed')
    payload = torch.load(best, map_location='cpu', weights_only=True, mmap=True)
    if payload['fingerprint'] != job['fingerprint']:
        raise ValueError('Checkpoint belongs to another task')
    model = build_model(local, cfg, device, load_parent=False)
    model.load_state_dict(payload['model'], strict=True)
    del payload
    return model, local, cfg, receipt


def package(values, rows, output: Path, name: str) -> Path:
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    validate_prediction(values, rows)
    atomic_json(output / 'prediction.json', values)
    path = output / name
    with zipfile.ZipFile(path.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as zipped:
        zipped.write(output / 'prediction.json', 'prediction.json')
    path.with_suffix('.zip.partial').replace(path)
    with zipfile.ZipFile(path) as zipped:
        if zipped.namelist() != ['prediction.json'] or zipped.testzip() is not None:
            raise ValueError('Submission ZIP verification failed')
    return path


# CELL: Raw 0.5 baseline, guarded calibration, frozen recipe then test predictions
def export(job_path):
    job = read_json(job_path)
    cfg = LSIConfig(**job['config'])
    root = Path(cfg.run_dir)
    if read_json(root / 'training_summary.json')['status'] != 'complete':
        raise ValueError('Complete/resume training before exporting a completed experiment')
    best = read_json(root / 'checkpoints/best.pt.json')
    directory = root / 'predictions'
    directory.mkdir(exist_ok=True)
    artifacts = {}
    all_logits = {}
    # Only larger A/C/D models can teach B. Avoid another full train-set pass for B.
    exported_splits = ('train', 'dev') if cfg.export_teacher_logits and cfg.family != 'b' else ('dev',)
''')


[Markdown Cell]

Importable source part 5

In [ ]:
# [Code Cell]
append_module('predict.py', r'''    parameter_report = parallel_predict(job_path, exported_splits)
    for split in exported_splits:
        rows = job['cache']['cases'][split]
        logits = np.load(directory / (split + '_logits.npy'), allow_pickle=False)
        all_logits[split] = logits
        raw = case_diagnostics([r['gold'] for r in rows], (probabilities(logits) >= .5).astype(int), rows)
        atomic_json(directory / (split + '_raw_metrics.json'), raw)
    rows = job['cache']['cases']['dev']
    gold = [r['gold'] for r in rows]
    raw_score = metrics(gold, (probabilities(all_logits['dev']) >= .5).astype(int))['macro_f1']
    history=read_json(root/'training_summary.json')['history']
    selected=next((h for h in history if h['epoch']==best['epoch']),None)
    reload_check={'actual_reloaded_raw_dev_macro_f1':raw_score,'saved_best_epoch_raw_dev_macro_f1':selected.get('raw_dev_macro_f1') if selected else None,
        'checkpoint_sha256':best['sha256'],'note':'Scores come from actual reloaded tensors; possible floating-point decision drift is reported, never replaced by history.'}
    atomic_json(root/'best_reload_report.json',reload_check)
    calibration = choose_threshold(probabilities(all_logits['dev']), rows, cfg.calibrate_threshold, cfg.calibration_margin, cfg.seed)
    atomic_json(root / 'decision_rule.json', {**calibration, 'checkpoint_sha256': best['sha256'], 'prepared_identity': job['prepared_identity']})
    dev_prediction = prediction(probabilities(all_logits['dev']), rows, calibration['thresholds'], dimension=job['label_schema']['dev_evaluator_dimension'])
    atomic_json(directory / 'dev_prediction.json', dev_prediction)
    official(Path(job['prepared_dir']), directory / 'dev_prediction.json', calibration['selected_dev_macro_f1'], directory)
    atomic_json(directory / 'dev_metrics.json', case_diagnostics(gold, (probabilities(all_logits['dev']) >= calibration['thresholds']).astype(int), rows))
    # The calibrated threshold is immutable before test logits are computed.
    rows = job['cache']['cases']['test']
    parallel_predict(job_path, ('test',))
    logits = np.load(directory / 'test_logits.npy', allow_pickle=False)
    final = prediction(probabilities(logits), rows, calibration['thresholds'])
    path = package(final, rows, root, 'submission_' + cfg.experiment_id + '.zip')
    # Preserve the raw decision rule as a separately named valid baseline.
    raw_dir = root / 'raw_baseline'
    raw_zip = package(prediction(probabilities(logits), rows), rows, raw_dir, 'submission_' + cfg.experiment_id + '_raw.zip')
    for split in (*exported_splits, 'test'):
        for suffix in ('_logits.npy', '_order.json'):
            name = split + suffix
            artifacts[name] = sha256_file(directory / name)
    receipt = {'status': 'submission_verified', 'version': 'lsi-output-v1', 'experiment_id': cfg.experiment_id,
               'family': cfg.family, 'method': cfg.method, 'label_count': cfg.label_count, 'prepared_identity': job['prepared_identity'],
               'fingerprint': job['fingerprint'], 'checkpoint_sha256': best['sha256'], 'checkpoint_epoch': best['epoch'],
               'raw_dev_macro_f1': raw_score, 'dev_macro_f1': calibration['selected_dev_macro_f1'],
               'threshold': calibration['thresholds'], 'calibration': calibration, 'parameter_report': parameter_report,
               'logit_artifacts': artifacts, 'zip_path': str(path), 'zip_sha256': sha256_file(path),
               'prediction_sha256': sha256_file(root / 'prediction.json'), 'raw_zip_path': str(raw_zip),
               'private_F1': None, 'test_labels_used': False}
    atomic_json(root / 'submission_receipt.json', receipt)
    return receipt


def main():
    parser = argparse.ArgumentParser()
''')


[Markdown Cell]

Importable source part 6

In [ ]:
# [Code Cell]
append_module('predict.py', r'''    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    print(export(args.job), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### preparation.py

Bounded JSON parsing and explicit field/layout overrides, repeated-ID audit, all source rows/views, public-only label schema and test annotation exclusion. A provided split overlap is recorded; dev independence is not assumed.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('preparation.py', r'''# CELL: Official LSI schemas and bounded streaming JSON readers
import json
import os
from pathlib import Path
import shutil
import sqlite3
import unicodedata
import re
import zipfile
from .io_utils import atomic_json, identity, read_json, sha256_file, records

FILES = {'train':'Phase_5_LSI/Train/train.json','dev':'Phase_5_LSI/Train/dev.json','test':'Phase_5_LSI/Test/test.json'}
IDS = ('id','case_id','doc_id','document_id')
TEXTS = ('text','document','case_text','content','paragraphs','sentences')
LABELS = ('labels','label','gold','statute_ids','statutes','applicable_statutes')
CONTAINERS = ('data','payload','cases','documents','docs','records','examples','items','dataset','train','dev','test')


def normalized_record(value, split, outer_id=None, fields=None, annotation_audit=None):
    """Only train/dev label lists are parsed; test projection contains ID and text."""
    fields = fields or {}
    if isinstance(value,str) and outer_id is not None:value={'id':outer_id,'text':value}
    if not isinstance(value,dict):raise ValueError('Each LSI case must be an object')
    get=lambda key,aliases,required=False:value[fields[key]] if key in fields else field(value,aliases,required)
    case_id=get('id',IDS)
    if case_id is None:case_id=outer_id
    if isinstance(case_id,int) and not isinstance(case_id,bool):case_id=str(case_id)
    if outer_id is not None and str(outer_id)!=case_id:raise ValueError('Outer key and case ID disagree')
    if not isinstance(case_id,str) or not case_id.strip():raise ValueError('Missing valid original case ID')
    text=flatten_text(get('text',TEXTS,True))
    row={'id':case_id,'text':text}
    if split=='test':
        if annotation_audit is not None:
            annotation_audit['records_with_annotation_keys']+=any(k in value for k in (*LABELS,'pred','target','targets','label'))
        return row
    labels=get('labels',LABELS,True)
    if not isinstance(labels,list) or any(type(x) is not int or x<0 for x in labels):
        raise ValueError(case_id+': labels must be a list of nonnegative integer statute IDs; set FIELD_MAP if needed')
    if len(set(labels))!=len(labels):raise ValueError(case_id+': duplicate label IDs; inspect source rather than inventing labels')
    row['gold']=sorted(labels)
    return row


def field(row: dict, names: tuple, required=False):
    found = [(key, row[key]) for key in names if key in row]
    if not found:
        if required:
            raise ValueError('Missing field: expected one of ' + ', '.join(names))
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('preparation.py', r'''        return None
    if any(value != found[0][1] for _, value in found[1:]):
        raise ValueError('Contradictory redundant fields: ' + ', '.join(k for k, _ in found))
    return found[0][1]


def flatten_text(value) -> str:
    """Preserve all original text/paragraph order; never shorten a case or split labels."""
    if isinstance(value, str):
        return value
    if isinstance(value, dict) and 'text' in value:
        return flatten_text(value['text'])
    if isinstance(value, list):
        return '\n\n'.join(flatten_text(part) for part in value)
    raise ValueError('Case text must be a string or ordered string/paragraph lists; set field overrides for a different schema')


def layout(path: Path, prefix='') -> tuple[str, str, dict]:
    """Inspect immediate fields with bounded metadata; named wrappers may be nested."""
    import ijson
    kind = None
    fields = {}
    with Path(path).open('rb') as stream:
        for current, event, value in ijson.parse(stream):
            if current == prefix and event in ('start_array', 'start_map') and kind is None:
                kind = event
                if kind == 'start_array':
                    return 'items', (prefix + '.item').lstrip('.'), {'kind': 'array', 'prefix': prefix}
            if current == prefix and event == 'map_key' and (len(fields) < 32 or value in CONTAINERS):
                fields[value] = None
            for key in tuple(fields):
                if fields[key] is None and current == (prefix + '.' + key).lstrip('.') and event not in ('map_key', 'end_map', 'end_array'):
                    fields[key] = event
    if kind != 'start_map':
        raise ValueError('Expected JSON array or object at prefix ' + prefix)
    containers = [key for key in CONTAINERS if fields.get(key) in ('start_array', 'start_map')]
    if any(fields.get(key) in ('string', 'number') for key in IDS) and any(key in fields for key in TEXTS):
        return 'single', prefix, {'kind': 'record', 'fields': fields, 'prefix': prefix}
    if len(containers) > 1:
        raise ValueError('Multiple document containers; set SOURCE_LAYOUTS explicitly: ' + repr(containers))
    if containers:
        return layout(path, (prefix + '.' + containers[0]).lstrip('.'))
    if any(key in fields for key in TEXTS):
        raise ValueError('Column-oriented/unknown JSON detected. Set SOURCE_LAYOUTS/FIELD_MAP explicitly; use record arrays or ID-record maps.')
    return 'mapping', prefix, {'kind': 'ID_mapping', 'fields': fields, 'prefix': prefix}


def stream_cases(path: Path, split: str, specification=None, fields=None, annotation_audit=None):
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('preparation.py', r'''    """At most one case object/text in memory; supported layouts never load the full array."""
    import ijson
    specification = specification or {}
    if specification:
        mode, prefix = specification['mode'], specification.get('prefix', '')
    else:
        mode, prefix, _ = layout(path)
    if mode not in ('items', 'mapping', 'single'):
        raise ValueError('SOURCE_LAYOUTS mode must be items, mapping or single')
    with Path(path).open('rb') as stream:
        if mode == 'mapping':
            for key, value in ijson.kvitems(stream, prefix):
                yield normalized_record(value, split, key, fields, annotation_audit)
        else:
            for value in ijson.items(stream, prefix):
                yield normalized_record(value, split, fields=fields, annotation_audit=annotation_audit)


# CELL: Only the three official LSI files, pinned to an immutable repository commit
def download_official(repository, revision, destination, token, path_map=None):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=token)
    try:
        commit = api.dataset_info(repository, revision=revision).sha
        available = set(api.list_repo_files(repository, repo_type='dataset', revision=commit))
    except Exception as error:
        raise RuntimeError('Enable Kaggle Internet and HF_TOKEN or HF Secret; accept the official dataset access terms') from error
    selected = {split:(path_map or {}).get(split,name) for split,name in FILES.items()}
    if set(selected.values()) - available:
        raise ValueError('Official LSI paths changed. Inspect the repository and set HF_PATH_MAP explicitly.')
    paths = {split:Path(hf_hub_download(repository, name, repo_type='dataset', revision=commit,
             token=token, local_dir=str(destination))) for split,name in selected.items()}
    return paths, {'repository':repository, 'commit':commit, 'paths':selected}


# CELL: Full source-row coverage and a taxonomy derived solely from public labels
def prepare(paths, output, provenance, evaluator, source_layouts=None, field_map=None, overlap_policy='warn'):
    """Stream records through SQLite. Retain every train/dev row and every test view.

    Repeated train/dev source IDs receive internal row keys and an explicit ID map.
    Repeated test IDs are one case with all its source text views, in source order.
    Test annotation values never enter prepared features, labels or the taxonomy.
    """
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    if overlap_policy not in ('warn','error'):raise ValueError('Overlap policy must be warn or error')
    choices={'policy':'lsi-public-labels-and-test-feature-allowlist-v1','provenance':provenance,
        'raw_sha256':{s:sha256_file(p) for s,p in paths.items()},'layouts':source_layouts or {},
        'fields':field_map or {},'overlap_policy':overlap_policy,'evaluator_sha256':sha256_file(evaluator)}
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('preparation.py', r'''    manifest_path=output/'prepared_manifest.json'
    if manifest_path.exists():
        prior=verify_prepared(output)
        if prior['source_identity']!=identity(choices):raise ValueError('Different preparation choices: use a new output directory')
        return prior
    database=output/'audit.sqlite'
    with sqlite3.connect(database) as db:
        db.execute('DROP TABLE IF EXISTS cases')
        db.execute('CREATE TABLE cases (split TEXT, seq INTEGER, id TEXT, text TEXT, norm TEXT, gold TEXT, PRIMARY KEY(split,seq))')
        counts={};all_labels=set();test_audit={'records_with_annotation_keys':0}
        for split in ('train','dev','test'):
            count=blank=chars=0
            for count,row in enumerate(stream_cases(paths[split],split,(source_layouts or {}).get(split),
                    (field_map or {}).get(split),test_audit),1):
                text=row['text'];norm=identity(re.sub(r'\s+',' ',unicodedata.normalize('NFKC',text)).strip())
                labels=row.get('gold')
                if labels is not None:all_labels.update(labels)
                db.execute('INSERT INTO cases VALUES (?,?,?,?,?,?)',(split,count,row['id'],text,norm,json.dumps(labels)))
                blank+=not text.strip();chars+=len(text)
                if count%1000==0:db.commit()
            if not count:raise ValueError(split+': no records found; check SOURCE_LAYOUTS/FIELD_MAP')
            counts[split]={'source_rows':count,'characters':chars,'blank_rows':blank}
        db.commit()
        db.execute('CREATE INDEX IF NOT EXISTS cases_split_id ON cases(split,id,seq)')
        db.execute('CREATE INDEX IF NOT EXISTS cases_norm ON cases(norm)')
        if not all_labels:raise ValueError('No public positive statute IDs were found')
        dimension=max(all_labels)+1
        if dimension>100000:raise ValueError('Unexpected statute dimension; inspect integer-ID schema before allocating a head')
        frequencies={split:[0]*dimension for split in ('train','dev')}
        artifacts={};policies={}
        for split in ('train','dev','test'):
            duplicate_ids={r[0] for r in db.execute('SELECT id FROM cases WHERE split=? GROUP BY id HAVING COUNT(*)>1',(split,))}
            path=output/(split+'.jsonl');mapping=output/(split+'_id_map.jsonl')
            n=0
            with path.with_suffix('.jsonl.partial').open('w',encoding='utf-8',newline='\n') as target, mapping.open('w',encoding='utf-8',newline='\n') as mapped:
                if split=='test':
                    iterator=db.execute('SELECT id,MIN(seq) FROM cases WHERE split=? GROUP BY id ORDER BY MIN(seq)',(split,))
                    for case_id,first in iterator:
                        views=[r[0] for r in db.execute('SELECT text FROM cases WHERE split=? AND id=? ORDER BY seq',(split,case_id))]
                        row={'id':case_id,'text':'\n\n'.join(views)}
                        target.write(json.dumps(row,ensure_ascii=False)+'\n');n+=1
                        mapped.write(json.dumps({'id':case_id,'source_id':case_id,'source_views':len(views)})+'\n')
                else:
                    for seq,case_id,text,labels in db.execute('SELECT seq,id,text,gold FROM cases WHERE split=? ORDER BY seq',(split,)):
                        key=('__lsi_'+split+'_row_'+identity([case_id,seq])) if case_id in duplicate_ids else case_id
                        row={'id':key,'source_id':case_id,'text':text,'gold':json.loads(labels)}
                        target.write(json.dumps(row,ensure_ascii=False)+'\n');n+=1
                        mapped.write(json.dumps({'id':key,'source_id':case_id,'source_row':seq})+'\n')
''')


[Markdown Cell]

Importable source part 5

In [ ]:
# [Code Cell]
append_module('preparation.py', r'''                        for label in row['gold']:frequencies[split][label]+=1
                target.flush();os.fsync(target.fileno())
            path.with_suffix('.jsonl.partial').replace(path)
            counts[split].update(cases=n,repeated_source_ids=len(duplicate_ids))
            policies[split]='All source rows retained' if split!='test' else 'One original ID, all text views joined in source order'
            for item in (path,mapping):artifacts[item.name]={'sha256':sha256_file(item),'bytes':item.stat().st_size}
        overlaps=[list(r) for r in db.execute('SELECT a.split,a.id,b.split,b.id FROM cases a JOIN cases b ON a.norm=b.norm AND a.split<b.split WHERE LENGTH(a.text)>=100 LIMIT 30')]
    schema={'label_count':dimension,'valid_label_ids':sorted(all_labels),
        'dev_evaluator_dimension':max((i for i,n in enumerate(frequencies['dev']) if n),default=-1)+1,
        'train_positive_counts':frequencies['train'],'dev_positive_counts':frequencies['dev'],
        'public_unobserved_columns':[i for i in range(dimension) if i not in all_labels],
        'dev_only_labels':[i for i in range(dimension) if frequencies['dev'][i] and not frequencies['train'][i]],
        'test_annotations_used':False}
    if not schema['dev_evaluator_dimension']:raise ValueError('Official local evaluator requires at least one dev positive label')
    atomic_json(output/'label_schema.json',schema)
    audit={'counts':counts,'row_policies':policies,'training_source_rows_dropped':0,
        'test_annotation_presence_only':test_audit,'test_gold_used':False,'cross_split_text_overlap_examples':overlaps,
        'validation_independence_warning':bool(overlaps),'overlap_policy':overlap_policy,
        'near_duplicate_scope':'Exact NFKC/whitespace-normalized full text only; semantic deduplication not claimed'}
    atomic_json(output/'data_audit.json',audit)
    if overlaps and overlap_policy=='error':raise ValueError('Official split text overlap found; see data_audit.json. No training rows were removed.')
    if overlaps:print('WARNING: source split text overlaps; dev scores may be optimistic. See data_audit.json.',flush=True)
    shutil.copyfile(evaluator,output/'evaluate_phase5.py')
    for name in ('label_schema.json','data_audit.json','evaluate_phase5.py'):
        p=output/name;artifacts[name]={'sha256':sha256_file(p),'bytes':p.stat().st_size}
    manifest={'version':'lsi-data-v1','status':'complete','source_identity':identity(choices),'source':choices,
        'counts':counts,'label_count':dimension,'artifacts':artifacts,'test_gold_used':False}
    manifest['identity']=identity(manifest);atomic_json(manifest_path,manifest)
    archive=output.parent/'phase5_prepared.zip'
    with zipfile.ZipFile(archive.with_suffix('.zip.partial'),'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for name in [*artifacts,'prepared_manifest.json']:z.write(output/name,name)
    archive.with_suffix('.zip.partial').replace(archive)
    return manifest


def verify_prepared(root):
    root=Path(root);m=read_json(root/'prepared_manifest.json')
    if m.get('version')!='lsi-data-v1' or m.get('status')!='complete' or identity({k:v for k,v in m.items() if k!='identity'})!=m['identity']:
        raise ValueError('Attach the complete Phase 5 LSI preparation output')
    for name,receipt in m['artifacts'].items():
        if Path(name).name!=name or '\\' in name or sha256_file(root/name)!=receipt['sha256'] or (root/name).stat().st_size!=receipt['bytes']:
            raise ValueError('Prepared artifact missing/changed: '+name)
    return m


def prepared_input(cfg):
    explicit=Path(cfg.prepared_dir) if cfg.prepared_dir else None
    if explicit and not explicit.exists():raise FileNotFoundError('PREPARED_DIR does not exist: '+str(explicit))
''')


[Markdown Cell]

Importable source part 6

In [ ]:
# [Code Cell]
append_module('preparation.py', r'''    roots=([explicit] if explicit.is_dir() else []) if explicit else [p.parent for p in Path(cfg.input_root).rglob('prepared_manifest.json') if read_json(p).get('version')=='lsi-data-v1']
    if not roots:
        archives=[explicit] if explicit else sorted(Path(cfg.input_root).rglob('phase5_prepared.zip'))
        for archive in archives:
            if not archive or not archive.is_file() or archive.suffix!='.zip':continue
            root=Path(cfg.run_dir)/'prepared_input';root.mkdir(parents=True,exist_ok=True)
            with zipfile.ZipFile(archive) as z:
                m=json.loads(z.read('prepared_manifest.json'));names=[*m['artifacts'],'prepared_manifest.json']
                if len(z.namelist())!=len(set(z.namelist())) or any(Path(n).name!=n or '\\' in n for n in names):raise ValueError('Unsafe prepared ZIP')
                for name in names:
                    with z.open(name) as source,(root/name).open('wb') as target:shutil.copyfileobj(source,target)
            roots.append(root)
    if not roots:raise FileNotFoundError('Attach Phase 5 notebook 00 complete output or phase5_prepared.zip')
    manifests=[verify_prepared(root) for root in roots]
    if len({m['identity'] for m in manifests})!=1:raise ValueError('Different prepared versions attached; set PREPARED_DIR explicitly')
    return roots[0]


''')


[Markdown Cell]

### prepare.py

Verify own Phase 1 source/checkpoint/data/tokenizer contracts. Resume full LSI state, not another task head. Distillation requires a completed larger LSI teacher and aligned training logits.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('prepare.py', r'''# CELL: Own-foundation verification and exact task recovery identities
from dataclasses import asdict, replace
from pathlib import Path
import shutil
import numpy as np
import torch
from . import VERSION
from .config import LSIConfig
from .data import cache_data, CaseDataset, collate, training_weights
from .io_utils import atomic_json, identity, read_json, sha256_file
from .parents import parent_class, source_hashes, tokenizer_inputs, verify_parent
from .preparation import prepared_input
from .state import Checkpoints


def require_hardware():
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2 or not torch.distributed.is_nccl_available():
        raise RuntimeError('Select Kaggle T4 x2: two CUDA devices and NCCL are required')
    for rank in range(2):
        name = torch.cuda.get_device_name(rank)
        if 'T4' not in name:
            raise RuntimeError('Expected T4 x2; found ' + name)
        print('GPU', rank, ':', name, flush=True)


def recovery_candidates(cfg):
    local = Path(cfg.run_dir) / 'job.json'
    paths = [local] if local.is_file() else ([Path(cfg.resume_run_dir) / 'job.json'] if cfg.resume_run_dir else sorted(Path(cfg.input_root).rglob('job.json')))
    result = []
    for path in paths:
        value = read_json(path)
        if value.get('config', {}).get('experiment_id') != cfg.experiment_id:
            if cfg.resume_run_dir:
                raise ValueError('Explicit resume root belongs to another experiment')
            continue
        if value.get('version') != VERSION or value['runtime_source_sha256'] != source_hashes(Path(__file__).parent):
            raise ValueError('Exact student recovery needs its original complete LSI notebook/runtime')
        if LSIConfig(**value['config']).choices() != cfg.choices():
            raise ValueError('Learning/head/chunk choices changed: use a new experiment ID, not an old optimizer')
        if value['environment']['torch'] != str(torch.__version__):
            raise ValueError('Exact recovery requires saved PyTorch build ' + value['environment']['torch'])
        if source_hashes(path.parent / 'code/lsi') != value['runtime_source_sha256']:
            raise ValueError('Saved student source files changed or are incomplete')
        result.append((path.parent, value))
    if len({value['fingerprint'] for _, value in result}) > 1:
        raise ValueError('Conflicting versions share this ID; set resume_run_dir to one exact task root')
    if result and not (Path(cfg.run_dir) / 'checkpoints').exists() and not any((root / 'checkpoints').is_dir() for root, _ in result):
        raise FileNotFoundError('The saved job has no actual checkpoint folder. Attach the full output, not observations.zip.')
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('prepare.py', r'''    return result


# CELL: Completed larger own LSI teachers, each checked against its own runtime
def teacher_inputs(job, cfg, catalog):
    destination = Path(cfg.run_dir) / 'teacher_cache.npy'
    reference_path = Path(cfg.run_dir) / 'teacher_reference.json'
    if destination.is_file() and reference_path.is_file():
        reference = read_json(reference_path)
        if sha256_file(destination) != reference['cache_sha256'] or reference['prepared_identity'] != job['prepared_identity']:
            raise ValueError('Saved teacher cache/data identity changed')
        return reference
    paths = [Path(cfg.teacher_run_dir) / 'job.json'] if cfg.teacher_run_dir else sorted(Path(cfg.input_root).rglob('job.json'))
    diagnostics, eligible = [], []
    expected_order = [{'id': r['id']} for r in job['cache']['cases']['train']]
    expected_dev = job['cache']['cases']['dev']
    from .metrics import metrics, probabilities
    for path in paths:
        row = {'task_root': str(path.parent), 'accepted': False, 'reasons': []}
        try:
            teacher = read_json(path)
            family = teacher.get('config', {}).get('family')
            if teacher.get('version') != VERSION or family not in ('a', 'c', 'd'):
                if cfg.teacher_run_dir:
                    raise ValueError('Explicit teacher is not a larger Phase 5 A/C/D task')
                continue
            root = path.parent
            receipt = read_json(root / 'submission_receipt.json')
            if read_json(root / 'training_summary.json')['status'] != 'complete' or receipt['status'] != 'submission_verified':
                raise ValueError('Teacher is paused or has not completed export')
            if receipt['prepared_identity'] != job['prepared_identity'] or receipt.get('label_count') != cfg.label_count:
                raise ValueError('Teacher data/outcome order differs')
            if receipt['fingerprint'] != teacher['fingerprint'] or teacher['prepared_identity'] != job['prepared_identity']:
                raise ValueError('Teacher job and submission receipt belong to different runs/data')
            if receipt['parameter_report']['foundation_unique_parameters'] <= job['parent_parameter_count']:
                raise ValueError('Teacher foundation must be larger than B for this KD experiment')
            if source_hashes(root / 'code/lsi') != teacher['runtime_source_sha256']:
                raise ValueError('Teacher code differs from its OWN saved source manifest')
            if (teacher['parent_contract']['source_sha256'] != catalog[family]['source_sha256']
                    or source_hashes(root / 'parent_source' / ('model_' + family)) != catalog[family]['source_sha256']):
                raise ValueError('Teacher does not use an approved own Phase 1 architecture')
            best = root / 'checkpoints/best.pt'
            checkpoint_receipt = read_json(root / 'checkpoints/best.pt.json')
            if (sha256_file(best) != receipt['checkpoint_sha256'] or checkpoint_receipt['sha256'] != receipt['checkpoint_sha256']
                    or checkpoint_receipt['fingerprint'] != teacher['fingerprint']):
                raise ValueError('Teacher checkpoint bytes and receipt disagree')
            for split, expected in (('train', expected_order), ('dev', [{'id': r['id']} for r in expected_dev])):
                for suffix in ('_logits.npy', '_order.json'):
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('prepare.py', r'''                    name = split + suffix
                    if sha256_file(root / 'predictions' / name) != receipt['logit_artifacts'][name]:
                        raise ValueError('Teacher exported logit/order checksum changed: ' + name)
                if read_json(root / 'predictions' / (split + '_order.json')) != expected:
                    raise ValueError('Teacher logits are not aligned to original case IDs')
                logits = np.load(root / 'predictions' / (split + '_logits.npy'), mmap_mode='r', allow_pickle=False)
                if logits.shape != (len(expected), cfg.label_count) or not np.isfinite(logits).all():
                    raise ValueError('Teacher logits are nonfinite or have wrong case/class dimensions')
            dev_logits = np.load(root / 'predictions/dev_logits.npy', allow_pickle=False)
            score = metrics([r['gold'] for r in expected_dev], (probabilities(dev_logits) >= .5).astype(int))['macro_f1']
            if abs(score - receipt['raw_dev_macro_f1']) > 1e-9:
                raise ValueError('Teacher exported dev logits do not reproduce its native F1')
            row['accepted'] = True
            row['raw_dev_macro_f1'] = score
            eligible.append((score, root, teacher, receipt))
        except (ValueError, KeyError, OSError) as error:
            row['reasons'].append(str(error))
        diagnostics.append(row)
    atomic_json(Path(cfg.run_dir) / 'teacher_input_report.json', diagnostics)
    print({'teacher_candidates': diagnostics}, flush=True)
    if not eligible:
        raise FileNotFoundError('No compatible COMPLETE larger PHASE 4 teacher. Attach full 01/02/04/05 LSI output with actual best.pt/receipt, source and train/dev logits; see teacher_input_report.json. Phase 3 models and observation ZIPs are not LSI teachers.')
    _, root, teacher, receipt = max(eligible, key=lambda x: (x[0], str(x[1])))
    values = np.load(root / 'predictions/train_logits.npy', allow_pickle=False)
    with destination.with_suffix('.npy.partial').open('wb') as stream:
        np.save(stream, values)
    destination.with_suffix('.npy.partial').replace(destination)
    reference = {'teacher_experiment': teacher['config']['experiment_id'], 'teacher_checkpoint_sha256': receipt['checkpoint_sha256'],
                 'teacher_fingerprint': teacher['fingerprint'], 'teacher_source_sha256': teacher['runtime_source_sha256'],
                 'teacher_raw_dev_macro_f1': receipt['raw_dev_macro_f1'], 'prepared_identity': job['prepared_identity'],
                 'label_count': cfg.label_count, 'cache_sha256': sha256_file(destination), 'training_only': True,
                 'teacher_foundation_parameters': receipt['parameter_report']['foundation_unique_parameters'],
                 'student_foundation_parameters': job['parent_parameter_count'], 'test_labels_used': False}
    atomic_json(reference_path, reference)
    return reference


# CELL: Portable full-state resumption and explicit total-epoch extension
def prepare_job(cfg, catalog):
    cfg.validate()
    run = Path(cfg.run_dir)
    run.mkdir(parents=True, exist_ok=True)
    prepared = prepared_input(cfg)
    manifest = read_json(prepared / 'prepared_manifest.json')
    schema = read_json(prepared / 'label_schema.json')
    if cfg.label_count not in (0,schema['label_count']):
        raise ValueError('Configured label dimension differs from prepared public taxonomy')
    cfg = replace(cfg,label_count=schema['label_count'])
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('prepare.py', r'''    candidates = recovery_candidates(cfg)
    prior = candidates[0] if candidates else None
    source = run / 'parent_source' / ('model_' + cfg.family)
    source.mkdir(parents=True, exist_ok=True)
    small = run / 'tokenizer'
    small.mkdir(exist_ok=True)
    contract = catalog[cfg.family]
    if prior:
        root, previous = prior
        parent = previous['parent']
        for name in contract['source_sha256']:
            origin = root / 'parent_source' / ('model_' + cfg.family) / name
            if sha256_file(origin) != contract['source_sha256'][name]:
                raise ValueError('Saved foundation source changed')
            if origin.resolve() != (source / name).resolve():
                shutil.copyfile(origin, source / name)
        for name, key in (('tokenizer.json', 'tokenizer_sha256'), ('metadata.json', 'metadata_sha256')):
            origin = root / 'tokenizer' / name
            if sha256_file(origin) != parent[key]:
                raise ValueError('Saved tokenizer snapshot changed')
            if origin.resolve() != (small / name).resolve():
                shutil.copyfile(origin, small / name)
        for name in ('teacher_cache.npy', 'teacher_reference.json'):
            origin = root / name
            if origin.is_file() and origin.resolve() != (run / name).resolve():
                shutil.copyfile(origin, run / name)
    else:
        root, parent = verify_parent(cfg, contract)
        token, meta, hashes = tokenizer_inputs(cfg, contract)
        parent.update(hashes)
        for name in contract['source_sha256']:
            shutil.copyfile(root / 'code' / ('model_' + cfg.family) / name, source / name)
        shutil.copyfile(token, small / 'tokenizer.json')
        shutil.copyfile(meta, small / 'metadata.json')
    if cfg.expected_parent_sha256 and parent['checkpoint_sha256'] != cfg.expected_parent_sha256:
        raise ValueError('Different selected Phase 1 checkpoint; use the audited one or explicitly accept another verified OWN continuation with expected_parent_sha256=None')
    cache = cache_data(prepared, small / 'tokenizer.json', cfg)
    Config, Model = parent_class(str(source), cfg.family, contract)
    with torch.device('meta'):
        probe = Model(Config(**parent['config']))
    parameters = sum(p.numel() for p in probe.parameters())
    assert 50_000_000 < parameters < 300_000_000
    del probe
    environment = {'torch': str(torch.__version__), 'numpy': np.__version__, 'tokenizers': __import__('tokenizers').__version__}
    runtime = source_hashes(Path(__file__).parent)
    job = {'version': VERSION, 'config': asdict(cfg), 'prepared_dir': str(prepared), 'prepared_identity': manifest['identity'],
           'parent': parent, 'parent_contract': contract, 'parent_source_dir': str(source),
           'parent_parameter_count': parameters, 'tokenizer_path': str(small / 'tokenizer.json'),
''')


[Markdown Cell]

Importable source part 5

In [ ]:
# [Code Cell]
append_module('prepare.py', r'''           'cache': cache, 'runtime_source_sha256': runtime, 'environment': environment,
           'resume_roots': [str(root) for root, _ in candidates], 'label_count': schema['label_count'], 'label_schema': schema}
    if cfg.method == 'kd':
        job['teacher'] = teacher_inputs(job, cfg, catalog)
    fingerprint = identity({'version': VERSION, 'choices': cfg.choices(), 'prepared': manifest['identity'],
                            'parent': parent['checkpoint_sha256'], 'tokenizer': parent['tokenizer_sha256'],
                            'runtime': runtime, 'environment': environment, 'cache': cache['identity'],
                            'case_metadata': cache['manifest_sha256'], 'teacher': job.get('teacher')})
    if prior and prior[1]['fingerprint'] != fingerprint:
        raise ValueError('Exact student recovery fingerprint changed')
    job['fingerprint'] = fingerprint
    manager = Checkpoints(run, fingerprint, cfg.min_free_bytes, job['resume_roots'])
    latest = manager.latest()
    effective, extending = cfg.max_epochs, False
    if latest:
        payload = manager.load(latest)
        original = payload['config']['max_epochs']
        effective = max(original, cfg.max_epochs)
        if cfg.target_total_epochs is not None:
            if cfg.target_total_epochs < payload['cursor']['epoch']:
                raise ValueError('Target total cannot be less than already completed epochs')
            effective = cfg.target_total_epochs
            extending = (cfg.target_total_epochs > payload['cursor']['epoch'] and cfg.target_total_epochs != payload['config'].get('target_total_epochs'))
        if payload['cursor']['complete'] and cfg.target_total_epochs is None and cfg.max_epochs > original:
            raise ValueError('Use TARGET_TOTAL_EPOCHS to extend a completed task; do not change only MAX_EPOCHS')
        schedule = prior[1].get('schedule_epochs', original)
        del payload
    else:
        if prior and prior[1].get('resume_checkpoint'):
            raise ValueError('Saved recovery checkpoint is missing; no silent fresh restart')
        schedule = cfg.max_epochs
        if cfg.target_total_epochs is not None:
            effective = cfg.target_total_epochs
    cfg = replace(cfg, max_epochs=effective)
    job.update(config=asdict(cfg), extension_requested=extending, schedule_epochs=schedule, resume_checkpoint=str(latest[0]) if latest else None)
    atomic_json(run / 'job.json', job)
    atomic_json(run / 'task_config.json', asdict(cfg))
    return job


def sanity(job, cfg):
    from .model import build_model, forward
    device = torch.device('cuda', 0)
    model = build_model(job, cfg, device, load_parent=not bool(job['resume_checkpoint']))
    if job['resume_checkpoint']:
        payload = torch.load(job['resume_checkpoint'], map_location='cpu', weights_only=True, mmap=True)
        model.load_state_dict(payload['model'], strict=True)
        del payload
''')


[Markdown Cell]

Importable source part 6

In [ ]:
# [Code Cell]
append_module('prepare.py', r'''    dataset = CaseDataset(job['cache'], 'train', cfg)
    index = min(range(len(dataset)), key=lambda i: dataset.rows[i]['length'])
    batch = collate([dataset[index], dataset[-1]])
    model.train()
    with torch.autocast('cuda', dtype=torch.float16):
        logits = forward(model, batch, device)
        from .train import objective
        loss = objective(logits,batch['targets'].to(device),torch.tensor(training_weights(job['cache'],cfg),device=device),batch['teacher'].to(device),cfg)
    if logits.shape != (2, cfg.label_count) or not torch.isfinite(logits).all() or not torch.isfinite(loss):
        raise ValueError('Full-size LSI model failed finite shape check')
    loss.backward()
    gradients = [p.grad for p in model.backbone.parameters() if p.requires_grad]
    if not gradients or not all(g is not None and torch.isfinite(g).all() for g in gradients):
        raise ValueError('Active backbone adaptation has missing/nonfinite gradients')
    if model.classifier.weight.grad is None or not torch.isfinite(model.classifier.weight.grad).all():
        raise ValueError('Classification head has missing/nonfinite gradients')
    weights = training_weights(job['cache'], cfg)
    result = {**model.parameter_report, 'logit_shape': list(logits.shape), 'finite_shape_and_gradients': True,
              'training_cases': len(dataset), 'training_class_weights': weights.tolist(),
              'chunk_counts_quantiles': np.quantile([r['chunks'] for r in dataset.rows], [0, .5, .9, .99, 1]).tolist(),
              'scope': 'Actual full architecture/starting tensors sanity; no model quality claim'}
    atomic_json(Path(cfg.run_dir) / 'sanity_report.json', result)
    dataset.close()
    del model, logits, loss
    torch.cuda.empty_cache()
    return result
''')


[Markdown Cell]

### profile.py

Measure discarded-weight full-architecture timings at length quantiles on two T4s. Report actual all-case work and a rough full-epoch range. This probe is not training on a sampled dataset, does not save learned weights and makes no F1 claim.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('profile.py', r'''# CELL: Discarded-weight performance probe; all-case epoch work is estimated explicitly
import math
from pathlib import Path
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from .config import LSIConfig
from .data import CaseDataset, collate, training_weights
from .io_utils import atomic_json, read_json
from .model import build_model, forward
from .state import seed_everything
from .train import objective, optimizer_for


def profile_worker(rank, job, port):
    cfg = LSIConfig(**job['config'])
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank, world_size=2)
    try:
        seed_everything(cfg.seed)
        model = build_model(job, cfg, device, load_parent=not bool(job['resume_checkpoint']))
        if job['resume_checkpoint']:
            model.load_state_dict(torch.load(job['resume_checkpoint'], map_location='cpu', weights_only=True, mmap=True)['model'])
        model = DDP(model, device_ids=[rank], broadcast_buffers=False)
        optimizer = optimizer_for(model, cfg)
        weights = torch.tensor(training_weights(job['cache'], cfg), device=device)
        dataset = CaseDataset(job['cache'], 'train', cfg)
        ordered = sorted(range(len(dataset)), key=lambda i: dataset.rows[i]['chunks'])
        # A performance probe only: original foundations/task checkpoints are never changed.
        selected = [ordered[round((len(ordered) - 1) * q)] for q in (.25, .5, .75, .9, .99)]
        measurements = []
        scaler = torch.amp.GradScaler('cuda', init_scale=4096)
        for index in selected:
            batch = collate([dataset[index]])
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            dist.barrier()
            torch.cuda.reset_peak_memory_stats()
            started = time.monotonic()
            with torch.autocast('cuda', dtype=torch.float16):
                logits = forward(model, batch, device)
                loss = objective(logits, batch['targets'].to(device), weights, batch['teacher'].to(device), cfg)
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('profile.py', r'''            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()
            stats = torch.tensor([time.monotonic() - started, torch.cuda.max_memory_allocated() / 2**30], device=device)
            dist.all_reduce(stats, op=dist.ReduceOp.MAX)
            measurements.append({'case_chunks': dataset.rows[index]['chunks'], 'seconds_per_case_with_update': float(stats[0]), 'peak_GiB_per_GPU': float(stats[1])})
        if rank == 0:
            chunks = sum(row['chunks'] for row in dataset.rows)
            ratios = [row['seconds_per_case_with_update'] / row['case_chunks'] for row in measurements[1:]]
            estimate = chunks * float(np.median(ratios)) / 2
            atomic_json(Path(cfg.run_dir) / 'profile_report.json', {
                'status': 'performance_probe_complete', 'training_cases': len(dataset), 'all_training_chunks': chunks,
                'measurements': measurements, 'rough_training_epoch_seconds': estimate,
                'rough_epoch_range_seconds': [estimate * .65, estimate * 2],
                'scope': 'Discarded-weight timings on length quantiles, not reduced-data training or F1. Excludes full dev/export/checkpoint time and worst-case length outliers. Measure the first real epoch before allocating all six runs.',
                'model_parameter_report': model.module.parameter_report})
        dataset.close()
    finally:
        dist.destroy_process_group()


def run_profile(job_path):
    job = read_json(job_path)
    if job['config']['method'] == 'kd':
        raise ValueError('Profile supervised B first; KD needs its completed larger teacher')
    with socket.socket() as listener:
        listener.bind(('127.0.0.1', 0))
        port = listener.getsockname()[1]
    mp.spawn(profile_worker, args=(job, port), nprocs=2, join=True)
    return read_json(Path(job['config']['run_dir']) / 'profile_report.json')
''')


[Markdown Cell]

### state.py

Two atomic recovery slots plus a separate best snapshot preserve model/Adam/scaler/scheduler and exact next case-update cursor. Checksummed receipts detect truncation; an intact older slot can recover. Preserve the full run output, not just a notebook or observations ZIP.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('state.py', r'''# CELL: Per-rank reproducibility and successful-update scheduling
import math
import os
from pathlib import Path
import random
import numpy as np
import torch
from .io_utils import atomic_json, ensure_space, publish_copy, read_json, sha256_file

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {'python':random.getstate(), 'numpy':[numpy_state[0],numpy_state[1].tolist(),*numpy_state[2:]],
            'torch':torch.get_rng_state(), 'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state['python'])
    value=state['numpy']; np.random.set_state((value[0],np.asarray(value[1],dtype=np.uint32),*value[2:]))
    torch.set_rng_state(state['torch'])
    if device.type=='cuda':
        torch.cuda.set_rng_state(state['cuda'],device)

class WarmupCosine:
    """Schedule advances after successful optimizer updates, including on resume."""
    def __init__(self, optimizer, total_steps: int, warmup_fraction: float, minimum: float):
        self.optimizer, self.total_steps, self.minimum = optimizer,total_steps,minimum
        self.warmup_steps=max(1,int(total_steps*warmup_fraction))
        self.base_lrs=[group['lr'] for group in optimizer.param_groups]; self.steps=0
        self.apply()

    def apply(self) -> None:
        step=self.steps+1
        if step <= self.warmup_steps:
            scale=step/self.warmup_steps
        else:
            progress=min(1.0,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))
            scale=self.minimum+(1-self.minimum)*0.5*(1+math.cos(math.pi*progress))
        for group,base in zip(self.optimizer.param_groups,self.base_lrs):
            group['lr']=base*scale

    def advance(self) -> None:
        self.steps+=1; self.apply()

''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('state.py', r'''    def state_dict(self) -> dict:
        return dict(steps=self.steps,total_steps=self.total_steps,warmup_steps=self.warmup_steps,
                    minimum=self.minimum,base_lrs=self.base_lrs)

    def load_state_dict(self, value: dict) -> None:
        for key in ('total_steps','warmup_steps','minimum','base_lrs'):
            if value[key] != getattr(self,key):
                raise ValueError(f'Scheduler plan changed: {key}')
        self.steps=value['steps']; self.apply()

# CELL: Two atomic full-state recovery slots plus a validation-best snapshot
class Checkpoints:
    """Commit receipts last; select the latest intact state, preserving read-only inputs."""
    def __init__(self, run: Path, fingerprint: str, reserve: int, extra_roots: list[str] | None = None):
        self.directory=run/'checkpoints'; self.directory.mkdir(parents=True,exist_ok=True)
        self.fingerprint,self.reserve,self.generation=fingerprint,reserve,0
        self.roots=[self.directory,*[Path(root)/'checkpoints' for root in (extra_roots or [])]]
        self.warnings=[]

    def candidates(self, best_only: bool = False) -> list[tuple[Path,dict]]:
        result,seen=[],False
        for root in self.roots:
            for name in (('best.pt',) if best_only else ('slot_0.pt','slot_1.pt','best.pt')):
                path=root/name; sidecar=root/(name+'.json')
                if not path.exists() and not sidecar.exists():
                    continue
                seen=True
                try:
                    receipt=read_json(sidecar)
                    if receipt.get('fingerprint') != self.fingerprint or receipt.get('version') != 1:
                        raise ValueError('Experiment/source/Torch fingerprint mismatch')
                    if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
                        raise ValueError('Checkpoint size/SHA mismatch')
                    result.append((path,receipt)); self.generation=max(self.generation,receipt['generation'])
                except (ValueError,KeyError,OSError) as error:
                    self.warnings.append(f'{path}: {error}')
        if seen and not result:
            raise ValueError('All visible LSI checkpoints failed verification. Refusing to silently restart. '+ '; '.join(self.warnings))
        return result

    def latest(self) -> tuple[Path,dict] | None:
        choices=self.candidates()
        if not choices:
            return None
        winner=max(choices,key=lambda pair:(pair[1]['step'],pair[1]['generation']))
        tied=[receipt['sha256'] for _,receipt in choices if (receipt['step'],receipt['generation'])==
              (winner[1]['step'],winner[1]['generation'])]
        if len(set(tied))>1:
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('state.py', r'''            raise ValueError('Conflicting recovery states at the same step. Attach one exact saved version.')
        return winner

    def best(self) -> tuple[Path,dict] | None:
        values=self.candidates(best_only=True)
        return max(values,key=lambda pair:(pair[1]['validation_macro_f1'],pair[1]['step'])) if values else None

    def adopt(self, candidate: tuple[Path,dict], best: bool = False) -> tuple[Path,dict]:
        source,receipt=candidate
        destination=self.directory/('best.pt' if best else f"slot_{receipt['generation']%2}.pt")
        if source.resolve()!=destination.resolve():
            ensure_space(self.directory,source.stat().st_size,self.reserve)
            publish_copy(source,destination)
            if sha256_file(destination)!=receipt['sha256']:
                raise ValueError('Restaged checkpoint checksum mismatch.')
            atomic_json(destination.with_name(destination.name+'.json'),receipt)
        return destination,receipt

    def load(self, candidate: tuple[Path,dict]) -> dict:
        path,receipt=candidate
        self.generation=max(self.generation,receipt['generation'])
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        required={'version','fingerprint','config','model','optimizer','scaler','scheduler','epoch','step','cursor','rng_by_rank'}
        if required-payload.keys() or payload['fingerprint']!=self.fingerprint:
            raise ValueError('Incomplete/incompatible LSI checkpoint.')
        cursor=payload['cursor']
        if payload['epoch']!=receipt['epoch'] or payload['step']!=receipt['step'] or cursor['global_step']!=payload['step']:
            raise ValueError('Checkpoint cursor and receipt disagree.')
        if payload['scheduler']['steps']!=cursor['optimizer_steps'] or len(payload['rng_by_rank'])!=payload['config']['world_size']:
            raise ValueError('Missing/misaligned scheduler or per-rank RNG state.')
        return payload

    def commit(self, payload: dict, best: bool = False) -> tuple[Path,dict]:
        self.generation+=1
        path=self.directory/f'slot_{self.generation%2}.pt'
        temporary=path.with_name(path.name+'.partial')
        estimate=sum(tensor.numel()*tensor.element_size() for tensor in payload['model'].values())*3+(16<<20)
        ensure_space(self.directory,estimate,self.reserve)
        with temporary.open('wb') as stream:
            torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary,path)
        receipt={'version':1,'fingerprint':self.fingerprint,'sha256':sha256_file(path),'bytes':path.stat().st_size,
                 'epoch':payload['epoch'],'step':payload['step'],'generation':self.generation,
                 'validation_macro_f1':payload['cursor']['best_macro_f1']}
        atomic_json(path.with_name(path.name+'.json'),receipt)
        if best:
            publish_copy(path,self.directory/'best.pt')
            atomic_json(self.directory/'best.pt.json',receipt)
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('state.py', r'''        return path,receipt
''')


[Markdown Cell]

### train.py

FP16 NCCL DDP with exact case-normalized multilabel BCE or asymmetric loss. Bernoulli KD preserves simultaneous positive statutes. All rows are covered once per completed epoch; checkpoints preserve cursor, Adam/scaler/scheduler and both RNG states.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('train.py', r'''# CELL: One global case-normalized supervised/KD loss, with successful-update scheduling
import argparse
from contextlib import nullcontext
from datetime import timedelta
import math
import os
from pathlib import Path
import signal
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.nn import functional as F
from tqdm.auto import tqdm
from .config import LSIConfig
from .data import CaseDataset, CasePlan, collate, training_weights
from .io_utils import atomic_json, read_json, run_lock
from .metrics import official
from .model import build_model, forward
from .performance import MicroBatches
from .predict import logits_partition, score_dev
from .state import Checkpoints, WarmupCosine, capture_rng, restore_rng, seed_everything


def objective(logits, targets, weights, teacher, cfg):
    """Summed per-case multilabel objective; all-reduce normalizes actual cases."""
    scores = logits.float()
    valid = (targets[:,0] != -100).float()
    labels = targets.clamp_min(0).float()
    element = F.binary_cross_entropy_with_logits(scores, labels, pos_weight=weights, reduction='none')
    if cfg.loss_type == 'asymmetric':
        negative_factor = torch.sigmoid(scores).detach().pow(cfg.negative_gamma)
        element = element * torch.where(labels > 0, torch.ones_like(element), negative_factor)
    supervised = (element.mean(1) * valid).sum()
    if cfg.method != 'kd':
        return supervised
    temperature = cfg.distillation_temperature
    probability = torch.sigmoid(teacher.float() / temperature)
    soft = F.binary_cross_entropy_with_logits(scores / temperature, probability, reduction='none')
    entropy = -(probability * probability.clamp_min(1e-7).log() + (1-probability) * (1-probability).clamp_min(1e-7).log())
    divergence = ((soft-entropy).mean(1) * valid).sum() * temperature ** 2
    return (1-cfg.distillation_weight)*supervised + cfg.distillation_weight*divergence


def optimizer_for(model, cfg):
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('train.py', r'''    groups = [{'params': [p for n, p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad], 'lr': cfg.backbone_lr},
              {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad], 'lr': cfg.head_lr}]
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay, fused=True)


def initial_cursor():
    return {'epoch': 0, 'next_batch': 0, 'global_step': 0, 'optimizer_steps': 0, 'history': [],
            'best_macro_f1': -1., 'best_macro_ap': -1., 'bad_epochs': 0, 'complete': False, 'loss_sum': 0., 'weight_sum': 0.,
            'cases': 0, 'chunks': 0, 'overflows': 0, 'consecutive_overflows': 0}


def save_all(rank, manager, model, optimizer, scaler, scheduler, cursor, cfg, job, device, best=False, paused=False):
    states = [None] * cfg.world_size if rank == 0 else None
    dist.gather_object(capture_rng(device), states, dst=0)
    if rank == 0:
        payload = {'version': 1, 'fingerprint': job['fingerprint'], 'config': job['config'],
                   'model': model.module.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                   'scheduler': scheduler.state_dict(), 'epoch': cursor['epoch'], 'step': cursor['global_step'],
                   'cursor': cursor, 'rng_by_rank': states}
        _, receipt = manager.commit(payload, best=best)
        summary = {'status': 'complete' if cursor['complete'] else ('paused_recoverable' if paused else 'in_progress'),
                   'epoch': cursor['epoch'], 'next_batch': cursor['next_batch'], 'step': cursor['global_step'],
                   'optimizer_steps': cursor['optimizer_steps'], 'best_dev_macro_f1': cursor['best_macro_f1'],
                   'history': cursor['history'], 'checkpoint_sha256': receipt['sha256'], 'fingerprint': job['fingerprint']}
        atomic_json(Path(cfg.run_dir) / 'training_summary.json', summary)
    dist.barrier()


def validation(rank, model, job, cfg, device):
    local = logits_partition(model, job, cfg, 'dev', device, rank, cfg.world_size)
    gathered = [None] * cfg.world_size if rank == 0 else None
    dist.gather_object(local, gathered, dst=0)
    value = None
    if rank == 0:
        joined = {key: values for shard in gathered for key, values in shard.items()}
        value, predictions = score_dev(joined, job)
        output = Path(cfg.run_dir)
        atomic_json(output / 'dev_prediction.json', predictions)
        official(Path(job['prepared_dir']), output / 'dev_prediction.json', value['macro_f1'], output)
        atomic_json(output / 'dev_metrics.json', value)
    wrapped = [value]
    dist.broadcast_object_list(wrapped, src=0)
    return wrapped[0]


# CELL: Two NCCL workers; exact next-case restart and collective timed-stop checkpoints
def worker(rank, job, port, event):
    cfg = LSIConfig(**job['config'])
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('train.py', r'''    cfg.validate()
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank,
                            world_size=2, timeout=timedelta(hours=2))
    try:
        seed_everything(cfg.seed)
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest = (Path(job['resume_checkpoint']), read_json(job['resume_checkpoint'] + '.json')) if job['resume_checkpoint'] else None
        payload = manager.load(latest) if latest else None
        model = build_model(job, cfg, device, load_parent=payload is None)
        if payload:
            model.load_state_dict(payload['model'], strict=True)
        ddp = DDP(model, device_ids=[rank], broadcast_buffers=False, find_unused_parameters=False,
                  gradient_as_bucket_view=True)
        optimizer = optimizer_for(model, cfg)
        teacher = np.load(Path(cfg.run_dir) / 'teacher_cache.npy', mmap_mode='r', allow_pickle=False) if cfg.method == 'kd' else None
        dataset = CaseDataset(job['cache'], 'train', cfg, teacher)
        steps = math.ceil(len(dataset) / (cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps))
        scheduler = WarmupCosine(optimizer, steps * job['schedule_epochs'], cfg.warmup_fraction, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler('cuda', init_scale=4096)
        cursor = initial_cursor()
        if payload:
            scheduler.total_steps = payload['scheduler']['total_steps']
            scheduler.warmup_steps = payload['scheduler']['warmup_steps']
            scheduler.minimum = payload['scheduler']['minimum']
            scheduler.base_lrs = payload['scheduler']['base_lrs']
            optimizer.load_state_dict(payload['optimizer'])
            scaler.load_state_dict(payload['scaler'])
            scheduler.load_state_dict(payload['scheduler'])
            cursor = payload['cursor']
            restore_rng(payload['rng_by_rank'][rank], device)
            del payload
            if cursor['global_step'] != cursor['epoch'] * steps + cursor['next_batch'] or not 0 <= cursor['next_batch'] <= steps:
                raise ValueError('Recovery cursor differs from the exhaustive whole-case plan')
        else:
            seed_everything(cfg.seed + rank)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
        if job['extension_requested'] and cursor['complete']:
            cursor['complete'] = False
            cursor['bad_epochs'] = 0
        if cursor['complete']:
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
            return
        weights_numpy = training_weights(job['cache'], cfg)
        weights = torch.tensor(weights_numpy, device=device)
        normalizers = np.asarray([1.] * len(dataset) + [0.])
''')


[Markdown Cell]

Importable source part 4

In [ ]:
# [Code Cell]
append_module('train.py', r'''        started = time.monotonic()
        for epoch in range(cursor['epoch'], cfg.max_epochs):
            epoch_started = time.monotonic()
            ddp.train()
            plan = CasePlan(dataset.rows, cfg, rank, epoch)
            progress = tqdm(total=steps, initial=cursor['next_batch'],
                            desc=f'{cfg.experiment_id} epoch {epoch + 1}/{cfg.max_epochs}', disable=rank != 0)
            with MicroBatches(dataset, plan, cursor['next_batch'], cfg) as batches:
                for step in range(cursor['next_batch'], steps):
                    stop = torch.tensor(int(event.is_set() or time.monotonic() - started >= cfg.max_run_seconds), device=device)
                    dist.all_reduce(stop, op=dist.ReduceOp.MAX)
                    if stop.item():
                        progress.close()
                        save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device, paused=True)
                        return
                    denominator = float(normalizers[plan.global_indices(step)].sum())
                    if denominator <= 0:
                        raise ValueError('A global update contains no real weighted cases')
                    optimizer.zero_grad(set_to_none=True)
                    stats = torch.zeros(3, device=device, dtype=torch.float64)
                    for micro in range(cfg.accumulation_steps):
                        batch = next(batches)
                        targets = batch['targets'].to(device, non_blocking=True)
                        context = ddp.no_sync() if micro + 1 < cfg.accumulation_steps else nullcontext()
                        with context:
                            with torch.autocast('cuda', dtype=torch.float16):
                                logits = forward(ddp, batch, device)
                                loss = objective(logits, targets, weights, batch['teacher'].to(device, non_blocking=True), cfg)
                            scaler.scale(loss * cfg.world_size / denominator).backward()
                        stats[0] += loss.detach().double()
                        stats[1] += (targets[:,0] != -100).sum()
                        stats[2] += batch['input_ids'].shape[0]
                    dist.all_reduce(stats)
                    loss_sum, case_count, chunk_count = stats.cpu().tolist()
                    if not all(math.isfinite(value) for value in (loss_sum, case_count, chunk_count)):
                        raise FloatingPointError('Nonfinite loss; resume the last committed checkpoint')
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(ddp.parameters(), cfg.max_grad_norm)
                    previous_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale() >= previous_scale:
                        scheduler.advance()
                        cursor['optimizer_steps'] += 1
                        cursor['consecutive_overflows'] = 0
                    else:
                        cursor['overflows'] += 1
                        cursor['consecutive_overflows'] += 1
''')


[Markdown Cell]

Importable source part 5

In [ ]:
# [Code Cell]
append_module('train.py', r'''                    if cursor['consecutive_overflows'] > cfg.max_consecutive_overflows:
                        raise FloatingPointError('Repeated AMP overflows; inspect committed state before continuing')
                    cursor['global_step'] += 1
                    cursor['next_batch'] = step + 1
                    cursor['loss_sum'] += loss_sum
                    cursor['weight_sum'] += denominator
                    cursor['cases'] += int(case_count)
                    cursor['chunks'] += int(chunk_count)
                    optimizer.zero_grad(set_to_none=True)
                    if rank == 0:
                        progress.update()
                        if cursor['global_step'] % cfg.log_every_steps == 0 or step + 1 == steps:
                            progress.set_postfix(loss=cursor['loss_sum'] / cursor['weight_sum'], lr=optimizer.param_groups[0]['lr'],
                                                 step=cursor['global_step'], cases=cursor['cases'], amp_skips=cursor['overflows'])
                    if cursor['global_step'] % 500 == 0:
                        save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
            progress.close()
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
            value = validation(rank, ddp.module, job, cfg, device)
            if cursor['cases'] != len(dataset):
                raise ValueError('Epoch failed exact once-per-original-case supervision')
            improved = (value['macro_f1'] > cursor['best_macro_f1'] + 1e-12 or
                (abs(value['macro_f1']-cursor['best_macro_f1'])<=1e-12 and value['macro_average_precision']>cursor.get('best_macro_ap',-1.)+1e-12))
            cursor['best_macro_f1'] = max(cursor['best_macro_f1'], value['macro_f1'])
            if improved:cursor['best_macro_ap']=value['macro_average_precision']
            cursor['bad_epochs'] = 0 if improved else cursor['bad_epochs'] + 1
            cursor['history'].append({'epoch': epoch + 1, 'weighted_objective': cursor['loss_sum'] / cursor['weight_sum'],
                                      'training_cases': cursor['cases'], 'training_chunks_with_masked_dummies': cursor['chunks'],
                                      'overflows': cursor['overflows'], 'epoch_seconds_in_this_session_including_validation': time.monotonic() - epoch_started, **value})
            cursor.update(epoch=epoch + 1, next_batch=0, loss_sum=0., weight_sum=0., cases=0, chunks=0, overflows=0)
            cursor['complete'] = cursor['epoch'] >= cfg.max_epochs or (cursor['epoch'] >= cfg.minimum_epochs and cursor['bad_epochs'] >= cfg.patience)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device, best=improved)
            if rank == 0:
                print({'epoch': cursor['epoch'], 'native_dev_macro_F1': value['macro_f1'], 'complete': cursor['complete']}, flush=True)
            if cursor['complete']:
                return
        dataset.close()
    finally:
        dist.destroy_process_group()


# CELL: Importable spawn entrypoint, never nn.DataParallel or notebook-local worker functions
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    job = read_json(args.job)
    cfg = LSIConfig(**job['config'])
''')


[Markdown Cell]

Importable source part 6

In [ ]:
# [Code Cell]
append_module('train.py', r'''    from .prepare import require_hardware
    require_hardware()
    cfg.validate()
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING', '1')
    with run_lock(Path(cfg.run_dir)):
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest, best = manager.latest(), manager.best()
        if best:
            manager.adopt(best, best=True)
        if latest:
            latest = manager.adopt(latest)
            job['resume_checkpoint'] = str(latest[0])
            print('Resume exact saved step', latest[1]['step'], 'epoch', latest[1]['epoch'], flush=True)
        else:
            job['resume_checkpoint'] = None
        context = mp.get_context('spawn')
        event = context.Event()
        signal.signal(signal.SIGINT, lambda *_: event.set())
        signal.signal(signal.SIGTERM, lambda *_: event.set())
        with socket.socket() as listener:
            listener.bind(('127.0.0.1', 0))
            port = listener.getsockname()[1]
        processes = mp.spawn(worker, args=(job, port, event), nprocs=2, join=False)
        while not processes.join(timeout=1):
            pass


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### workflow.py

Automatic input checks, actual full-size shape/gradient sanity, training, completion-aware export and small observations. Paused or failed runs cannot claim a final submission. Keep full saved Kaggle output for continuation or KD.

[Markdown Cell]

Importable source part 1

In [ ]:
# [Code Cell]
append_module('workflow.py', r'''# CELL: Explicit completion, timed recovery and always-available observations
import argparse
from dataclasses import asdict
from pathlib import Path
import subprocess
import signal
import sys
import traceback
import zipfile
from .config import LSIConfig
from .io_utils import atomic_json, read_json
from .prepare import prepare_job, require_hardware, sanity
from .state import Checkpoints


def run_training(command: list[str]) -> None:
    """Forward a notebook stop to the trainer, then wait for its safe checkpoint.

    The workflow must not exit on SIGINT while its spawned trainer keeps running.
    Only the trainer coordinator receives the request; its workers synchronize
    the stop at the next global-update boundary before saving their state.
    """
    process=subprocess.Popen(command)
    previous={sig:signal.getsignal(sig) for sig in (signal.SIGINT,signal.SIGTERM)}
    def forward_stop(signum,frame):
        if process.poll() is None:process.send_signal(signum)
    try:
        for sig in previous:signal.signal(sig,forward_stop)
        status=process.wait()
        if status:raise subprocess.CalledProcessError(status,command)
    finally:
        for sig,handler in previous.items():signal.signal(sig,handler)


def stage_completed_export(job):
    """Publish a read-only completed input into the new writable replay run."""
    cfg = LSIConfig(**job['config'])
    root = Path(cfg.run_dir)
    manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    latest, best = manager.latest(), manager.best()
    if not latest or not best:
        raise FileNotFoundError('Export requires an actual complete latest checkpoint and best checkpoint')
    payload = manager.load(latest)
    cursor = payload['cursor']
    if not cursor['complete']:
        raise ValueError('Export cannot bypass an incomplete/paused training cursor')
    manager.adopt(best, best=True)
    latest = manager.adopt(latest)
''')


[Markdown Cell]

Importable source part 2

In [ ]:
# [Code Cell]
append_module('workflow.py', r'''    atomic_json(root / 'training_summary.json', {
        'status': 'complete', 'epoch': cursor['epoch'], 'next_batch': cursor['next_batch'],
        'step': cursor['global_step'], 'optimizer_steps': cursor['optimizer_steps'],
        'best_dev_macro_f1': cursor['best_macro_f1'], 'history': cursor['history'],
        'checkpoint_sha256': latest[1]['sha256'], 'fingerprint': job['fingerprint']})


def observations(root: Path):
    """Small diagnosis archive; deliberately not a replacement for model weights."""
    root.mkdir(parents=True, exist_ok=True)
    path = root / 'observations.zip'
    with zipfile.ZipFile(path.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as zipped:
        for item in sorted(root.rglob('*')):
            if item.is_file() and item.suffix in ('.json', '.log', '.txt') and item.stat().st_size <= 5 << 20:
                zipped.write(item, str(item.relative_to(root)))
    path.with_suffix('.zip.partial').replace(path)


def execute(plan_path, catalog_path):
    plan = read_json(plan_path)
    cfg = LSIConfig(**plan['config'])
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    try:
        # Find/validate all input identities before the expensive full-size GPU checks.
        job = prepare_job(cfg, read_json(catalog_path))
        atomic_json(root / 'input_report.json', {
            'prepared': job['prepared_dir'], 'parent': job['parent']['checkpoint_sha256'],
            'training_cases': len(job['cache']['cases']['train']),
            'all_training_chunks': sum(r['chunks'] for r in job['cache']['cases']['train']),
            'resume_checkpoint': job['resume_checkpoint'], 'teacher': job.get('teacher'),
            'learning_choices': LSIConfig(**job['config']).choices()})
        require_hardware()
        if plan.get('mode', 'train') == 'profile':
            from .profile import run_profile
            return run_profile(root / 'job.json')
        if plan.get('mode', 'train') not in ('train', 'export'):
            raise ValueError('Mode must be train, profile or export')
        if plan.get('mode', 'train') == 'train':
            sanity(job, LSIConfig(**job['config']))
            run_training([sys.executable, '-u', '-m', 'lsi.train', '--job', str(root / 'job.json')])
        else:
            stage_completed_export(job)
        if read_json(root / 'training_summary.json')['status'] != 'complete':
            result = {'status': 'paused_recoverable', 'next_action': 'Save the complete output and attach it to this same notebook; resume the exact cursor. No final ZIP is claimed.'}
        else:
            from .predict import export
            result = export(root / 'job.json')
''')


[Markdown Cell]

Importable source part 3

In [ ]:
# [Code Cell]
append_module('workflow.py', r'''        atomic_json(root / 'final_report.json', result)
        return result
    except Exception as error:
        atomic_json(root / 'failure_report.json', {'status': 'failed', 'type': type(error).__name__,
            'message': str(error), 'traceback': traceback.format_exc(), 'action': 'Keep full checkpoints and consult input/teacher/failure reports; never delete committed recovery files.'})
        raise
    finally:
        observations(root)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    parser.add_argument('--catalog', required=True)
    args = parser.parse_args()
    print(execute(args.plan, args.catalog), flush=True)


if __name__ == '__main__':
    main()
''')


In [ ]:
# [Code Cell]
EXPECTED_SOURCE_SHA256={'__init__.py': '747ede88378f99f8b7e4dfdb119bbd1dc91f64e1b9d87fe2448d3986e78492e9', 'adaptation.py': 'c34aac82fb4925398f3c6291f84206e65fe77c5a5e557d80baee756e6811ee78', 'analysis.py': '46ffa271256062ddd57f1075fa7833ef4cd8947d9ac5a88d3d9ffec51e88c7b5', 'compare.py': '1045c8162f6c9abe689cc66d10e2b9110ac69671cfcb4cff58026ca0c5b7a420', 'config.py': '7eaf0095f34bba761bb64e9ddfdba2548cb8034629b3f464d08fbf058288e4ea', 'data.py': '293b96f1a142c3efafd58a1e878ea5a7928d91a456a2c2ec3727237e178dd257', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': '7dc410eff0eee5a574b02fad473c31677e7aaf82bf57f02b382f026b5c4c989f', 'model.py': '76b9ea3b4740a878d8022a5f2f3f0aead187ba844080d96cb867d7c19b23c056', 'parents.py': 'ab848a5736f1c2e00c239fa126ed989d7c6d638f876f38850e31b8a59dbd8183', 'performance.py': 'bd3bfbaa476107db8c1896dbe5842055af2038dc0de36c8e427f0ee710f40ebf', 'predict.py': '5b5d22f04a38bb82917e385a6c29a594d7774e7fe29cf5eb6fc1468d54b89223', 'preparation.py': 'c77b12a3aa0fcfc6362a72caa36253c665cf4c084409ad597b3e99bc791cc6fd', 'prepare.py': '7dc28e20739bf5ba7c0b0871ad016e68e37c21178b51d1e0c46c8c21bc7d500c', 'profile.py': '8688a2fdbe91e40201c9fe166f24b246b383d3254a2934d635698f85481276ac', 'state.py': 'faf27539cf9310df766b946a96791ea7d0539272fbbc4612b7202841209678f2', 'train.py': '2e4ecdf8de9b38d7c03ee748838ed8df2247404c3eeb079bc1a2dae281176d88', 'workflow.py': 'c95d0ca1c289a8cebe4d7357e5275fbe4698d32811d1da53ac8944c84a794308'}


[Markdown Cell]

## 4. Materialize exact runtime and validated launch settings

In [ ]:
# [Code Cell]
import hashlib
import importlib
code_directory=Path(OUTPUT_ROOT)/'code'
package_directory=code_directory/'lsi'
package_directory.mkdir(parents=True,exist_ok=True)
for name,parts in module_parts.items():
    source=''.join(parts)
    assert hashlib.sha256(source.encode()).hexdigest()==EXPECTED_SOURCE_SHA256[name],name
    compile(source,name,'exec')
    destination=package_directory/name
    if (Path(OUTPUT_ROOT)/'job.json').exists() and destination.exists() and destination.read_bytes()!=source.encode():
        raise ValueError('Saved task runtime changed: use its exact notebook, or use a new experiment ID.')
    destination.write_bytes(source.encode())
for name in list(sys.modules):
    if name=='lsi' or name.startswith('lsi.'):
        del sys.modules[name]
sys.path.insert(0,str(code_directory))
importlib.invalidate_caches()


[Markdown Cell]

### Run-local input compatibility helpers
These helpers live outside the immutable `code/lsi` package. The teacher is read as validated numeric
outputs; its Python source is checked, not executed. The known numeric NumPy scalar helper keeps
`weights_only=True`. Source hashes below are checked before importing the helper.

In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS = {}


[Markdown Cell]

Helper `cs781_native_contracts.py`

In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('cs781_native_contracts.py', []).append(r'''"""Reviewed own-foundation portable structural fingerprints; Python 3.10-3.13."""
APPROVED_NATIVE_SOURCES = [{'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': '586cd48bdc420d8fed5db7b074ceba38491b8f091f8f28d702ed74983c000b63',
                      'model.py': '836d92b81250ca5d0e1a84c3df8181c1129e437bfea75c4f8653d2ebd3bf545b'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': 'bce55e18e1388720b701933df11a420783e56fe826a6780cfedce2551f468910',
                      'model.py': '43d74a2894b9bf74eee67f044a344557c39776a270491cd03a80e0440c44f1f4'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/Fast_40h/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': '6b6759c5c8375120cd82882ba4a2c48ef7ab4160895379c5ec29144f29551a52',
                      'model.py': '8af20d2199c02557c5f4c7b7f3ddf08a8dd3f0e3941bcb75bcda21761bc6e88e'},
  'package': 'a_upgrade',
  'profile': 'Foundation_Upgrade/ModelA/Fast_AutoResume_v3/runtime/a_upgrade',
  'schema': 'a-upgraded-foundation-v1'},
 {'algorithm': 'python-canonical-ast-v1',
  'core_ast_sha256': {'__init__.py': 'bf533c9507e9e23be619158713603433a021b6618ad714658f19edc0d45b9f7b',
                      'config.py': 'b83d3fee697e9eee3861de2d5d0e386a5b07a3397e3b80073aa58c1bdd1b0150',
                      'loss.py': 'd803e39be8f8998383b5627e711dda1a427f95cf358af7a0a9a58519cb38e92c',
                      'model.py': '9a90daad6e2cb3544d94ad14fbaa2df5453fd15250e7a7aa2ee45c51a4cc1025',
                      'model_b.py': '80f26dd23889221032a5444b37457aae0419536b7c498de328dbd675d9f9fb15',
                      'model_c.py': '0e1f50425fdcde72c90745888212fe305e898dcd844c246bab6a669d44316158',
                      'model_d.py': '8e768f3e73f78cf3e69347eae5972956fcfd10005e6849f564fef3be265f5b7f'},
  'package': 'foundation_upgrade',
  'profile': 'Foundation_Upgrade/Models_BCD/runtime/foundation_upgrade',
  'schema': 'bcd-upgraded-foundation-v1'}]
''')


[Markdown Cell]

Helper `cs781_numpy_resume.py`

In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('cs781_numpy_resume.py', []).append(r'''"""Read existing scalar-metadata checkpoints without changing the LSI runtime.

Only NumPy's numeric scalar constructor, dtype constructor and a fixed list of
numeric dtype classes are registered. Tensor/state loading stays weights_only.
No checkpoint is rewritten, and no globals are inferred from checkpoint text.
"""
from importlib import import_module
import io
import os

import numpy as np
import torch

VERSION = 'cs781-numpy-scalar-resume-v1'
DTYPE_NAMES = (
    'bool', 'int8', 'int16', 'int32', 'int64',
    'uint8', 'uint16', 'uint32', 'uint64',
    'float16', 'float32', 'float64',
)


def register_numpy_scalars() -> list[str]:
    """Allow the known numeric metadata types in this interpreter, idempotently."""
    namespace = 'numpy._core.multiarray' if int(np.__version__.split('.')[0]) >= 2 else 'numpy.core.multiarray'
    scalar_constructor = import_module(namespace).scalar
    accepted = [scalar_constructor, np.dtype, *[type(np.dtype(name)) for name in DTYPE_NAMES]]
    existing = torch.serialization.get_safe_globals()
    torch.serialization.add_safe_globals([value for value in accepted if value not in existing])
    os.environ['CS781_NUMPY_RESUME_PID'] = str(os.getpid())
    return [value.__module__ + '.' + value.__name__ for value in accepted]


def self_test() -> dict:
    """Exercise scalar loading on CPU without advancing any training RNG."""
    metadata = {name: np.dtype(name).type(1) for name in DTYPE_NAMES}
    stream = io.BytesIO()
    torch.save({'metadata': metadata, 'weights': torch.tensor([1., 2.])}, stream)
    stream.seek(0)
    loaded = torch.load(stream, map_location='cpu', weights_only=True)
    for name, original in metadata.items():
''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('cs781_numpy_resume.py', []).append(r'''        assert type(loaded['metadata'][name]) is type(original), name
        assert loaded['metadata'][name] == original, name
    assert torch.equal(loaded['weights'], torch.tensor([1., 2.]))
    return {'status': 'passed', 'compatibility_version': VERSION,
            'torch': str(torch.__version__), 'numpy': np.__version__,
            'weights_only': True, 'numeric_dtype_types': len(DTYPE_NAMES)}
''')


[Markdown Cell]

Helper `cs781_source_identity.py`

In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('cs781_source_identity.py', []).append(r'''"""Portable structural source identity, independent of ast.dump display changes."""
import ast
import hashlib
import json

ALGORITHM = 'python-canonical-ast-v1'


def canonical_tree(value):
    """Keep meaningful node fields; omit absent/default optional fields consistently.

    Python 3.12 added empty type_params; Python 3.13 changed ast.dump's default
    omission of empty lists. We never hash that debugging representation. Real
    nonempty fields, including generic type parameters, remain in the identity.
    Source locations, whitespace and comments do not affect model semantics.
    """
    if isinstance(value, ast.AST):
        fields = {}
        for name, item in ast.iter_fields(value):
            if item is None or (isinstance(item, list) and not item):
                continue
            fields[name] = canonical_tree(item)
        return {'node': type(value).__name__, 'fields': fields}
    if isinstance(value, list):
        return [canonical_tree(item) for item in value]
    if isinstance(value, bytes):
        return {'constant_type': 'bytes', 'hex': value.hex()}
    if value is Ellipsis:
        return {'constant_type': 'ellipsis'}
    if isinstance(value, complex):
        return {'constant_type': 'complex', 'real': value.real, 'imag': value.imag}
    return value


def source_fingerprint(source: str) -> str:
    """Hash a deterministic JSON representation, not Python-version-specific text."""
    payload = {'algorithm': ALGORITHM, 'tree': canonical_tree(ast.parse(source))}
    serialized = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(serialized.encode('ascii')).hexdigest()
''')


[Markdown Cell]

Helper `p5_teacher_compat.py`

In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('p5_teacher_compat.py', []).append(r'''"""Phase 5 KD input repair; original student runtime and checkpoints stay intact.

Teacher outputs are data: validate their own source/checkpoint/logit receipts,
then copy only official-training logits. Never import or execute teacher code.
"""
import argparse
import json
from pathlib import Path
import shutil

import numpy as np

SUPPORTED = {'cs781-lsi-v1', 'cs781-lsi-upgraded-v1'}


def _read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def _native_source(directory, family, contract):
    from cs781_native_contracts import APPROVED_NATIVE_SOURCES
    from cs781_source_identity import source_fingerprint
    from lsi.parents import source_hashes
    if source_hashes(directory) != contract['source_sha256']:
        raise ValueError('Native teacher foundation source bytes changed')
    schema = 'a-upgraded-foundation-v1' if family == 'a' else 'bcd-upgraded-foundation-v1'
    if contract.get('schema') != schema or contract.get('family') != family:
        raise ValueError('Native teacher foundation family/schema mismatch')
    for profile in APPROVED_NATIVE_SOURCES:
        if profile['schema'] == schema:
            actual = {name: source_fingerprint((directory / name).read_text(encoding='utf-8'))
                      for name in profile['core_ast_sha256']}
            if actual == profile['core_ast_sha256']:
                return
    raise ValueError('Unreviewed teacher foundation implementation; preserve its source for review')


def _paths(cfg):
    if cfg.teacher_run_dir:
        selected = Path(cfg.teacher_run_dir)
''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('p5_teacher_compat.py', []).append(r'''        return [selected if selected.name == 'job.json' else selected / 'job.json']
    return sorted(Path(cfg.input_root).rglob('job.json'))


def _reuse(root, identity, dimension, train_count):
    from lsi.io_utils import sha256_file
    root = Path(root)
    cache, receipt = root / 'teacher_cache.npy', root / 'teacher_reference.json'
    if not cache.is_file() or not receipt.is_file():
        return None
    reference = _read(receipt)
    if (reference['prepared_identity'] != identity or reference['label_count'] != dimension
            or sha256_file(cache) != reference['cache_sha256']):
        raise ValueError('Saved teacher cache identity/hash changed; keep the original cache and receipt together')
    if reference.get('teacher_loader_sha256') and reference['teacher_loader_sha256'] != sha256_file(Path(__file__)):
        raise ValueError('Resume using the same repaired notebook that created this teacher cache')
    values = np.load(cache, mmap_mode='r', allow_pickle=False)
    if values.shape != (train_count, dimension) or not np.isfinite(values).all():
        raise ValueError('Saved teacher cache has invalid dimensions or nonfinite values')
    return reference


def _validate(root, job, cfg, catalog):
    """Validate both legacy and upgraded teachers without weakening provenance."""
    from lsi.io_utils import sha256_file
    from lsi.parents import source_hashes
    from lsi.metrics import metrics, probabilities
    teacher = _read(root / 'job.json')
    family = teacher.get('config', {}).get('family')
    if teacher.get('version') not in SUPPORTED or family not in ('a', 'c', 'd'):
        raise ValueError('Need a completed Phase 5 A/C/D LSI task; a foundation or another phase is not a teacher')
    receipt = _read(root / 'submission_receipt.json')
    if (_read(root / 'training_summary.json')['status'] != 'complete'
            or receipt.get('status') != 'submission_verified'
            or receipt.get('version') != 'lsi-output-v1'):
        raise ValueError('Teacher must finish training and Phase 5 export first')
    if (teacher['prepared_identity'] != job['prepared_identity']
            or receipt['prepared_identity'] != job['prepared_identity']
            or receipt['label_count'] != cfg.label_count
            or receipt['fingerprint'] != teacher['fingerprint']):
''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('p5_teacher_compat.py', []).append(r'''        raise ValueError('Teacher data, label taxonomy, job and submission receipts do not match')
    count = receipt['parameter_report']['foundation_unique_parameters']
    if count <= job.get('parent_parameter_count', 0):
        raise ValueError('This KD experiment needs an A/C/D foundation larger than the selected B student')
    if source_hashes(root / 'code/lsi') != teacher['runtime_source_sha256']:
        raise ValueError('Teacher runtime differs from its own saved source manifest')
    source = root / 'parent_source' / ('model_' + family)
    contract = teacher['parent_contract']
    if contract.get('kind') == 'native_foundation_export':
        _native_source(source, family, contract)
    elif (contract['source_sha256'] != catalog[family]['source_sha256']
          or source_hashes(source) != catalog[family]['source_sha256']):
        raise ValueError('Teacher does not use the matching approved own Phase 1 architecture')
    checkpoint = _read(root / 'checkpoints/best.pt.json')
    if (sha256_file(root / 'checkpoints/best.pt') != receipt['checkpoint_sha256']
            or checkpoint['sha256'] != receipt['checkpoint_sha256']
            or checkpoint['fingerprint'] != teacher['fingerprint']):
        raise ValueError('Teacher checkpoint and its receipts disagree')
    for split in ('train', 'dev'):
        expected = [{'id': row['id']} for row in job['cache']['cases'][split]]
        for suffix in ('_logits.npy', '_order.json'):
            name = split + suffix
            file = root / 'predictions' / name
            if not file.is_file() or name not in receipt['logit_artifacts']:
                raise FileNotFoundError('Missing ' + name + ': export this completed teacher with MODE="export", '
                    'EXPORT_TRAIN_LOGITS=True in its exact matching training/export notebook. Do not retrain the teacher.')
            if sha256_file(file) != receipt['logit_artifacts'][name]:
                raise ValueError('Teacher logit/order checksum changed: ' + name)
        if _read(root / 'predictions' / (split + '_order.json')) != expected:
            raise ValueError('Teacher logits are not aligned to the official training/development IDs')
        values = np.load(root / 'predictions' / (split + '_logits.npy'), mmap_mode='r', allow_pickle=False)
        if values.shape != (len(expected), cfg.label_count) or not np.isfinite(values).all():
            raise ValueError('Wrong teacher logit dimensions or nonfinite values')
    dev = np.load(root / 'predictions/dev_logits.npy', mmap_mode='r', allow_pickle=False)
    score = metrics([row['gold'] for row in job['cache']['cases']['dev']],
                    (probabilities(dev) >= .5).astype(np.uint8))['macro_f1']
    if abs(score - receipt['raw_dev_macro_f1']) > 1e-9:
        raise ValueError('Teacher logits do not reproduce its saved raw development F1')
    return score, teacher, receipt

''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('p5_teacher_compat.py', []).append(r'''
def select_teacher(job, cfg, catalog):
    """Give explicit diagnostics before tokenization, never silently remove KD."""
    from lsi.io_utils import atomic_json
    candidates, diagnostics = [], []
    for path in _paths(cfg):
        row = {'task_root': str(path.parent), 'accepted': False, 'reasons': []}
        try:
            value = _read(path)
            if not cfg.teacher_run_dir and value.get('config', {}).get('family') not in ('a', 'c', 'd'):
                continue
            score, teacher, receipt = _validate(path.parent, job, cfg, catalog)
            row.update(accepted=True, version=teacher['version'], raw_dev_macro_f1=score)
            candidates.append((score, str(path.parent), teacher, receipt))
        except (OSError, ValueError, KeyError, TypeError, SyntaxError) as error:
            row['reasons'].append(str(error))
        diagnostics.append(row)
    atomic_json(Path(cfg.run_dir) / 'teacher_input_report.json', diagnostics)
    print(json.dumps({'teacher_candidates': diagnostics}, indent=2), flush=True)
    if not candidates:
        raise FileNotFoundError('No usable Phase 5 teacher output attached. Add the COMPLETE saved output of '
            'A Full / A LoRMA / C Adapters / D Partial (A Full is a convenient choice), containing job.json, '
            'checkpoints/best.pt and receipt, code/lsi, parent_source, submission_receipt.json and '
            'predictions/train_logits.npy, train_order.json, dev_logits.npy, dev_order.json. '
            'An .ipynb, observation ZIP, submission ZIP or Phase 1 model alone is insufficient. '
            'Set TEACHER_RUN_DIR to that exact task root. See teacher_input_report.json for rejection details.')
    return max(candidates, key=lambda candidate: (candidate[0], candidate[1]))


def teacher_inputs(job, cfg, catalog):
    """Install verified numeric targets, preserving old reference schemas on resume."""
    from lsi.io_utils import atomic_json, sha256_file
    root = Path(cfg.run_dir)
    reference = _reuse(root, job['prepared_identity'], cfg.label_count, len(job['cache']['cases']['train']))
    if reference is not None:
        return reference
    _, selected, teacher, receipt = select_teacher(job, cfg, catalog)
    selected = Path(selected)
    destination = root / 'teacher_cache.npy'
    temporary = destination.with_suffix('.npy.partial')
''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('p5_teacher_compat.py', []).append(r'''    shutil.copyfile(selected / 'predictions/train_logits.npy', temporary)
    if sha256_file(temporary) != receipt['logit_artifacts']['train_logits.npy']:
        raise ValueError('Teacher cache copy failed checksum verification')
    temporary.replace(destination)
    reference = {'teacher_experiment': teacher['config']['experiment_id'],
        'teacher_checkpoint_sha256': receipt['checkpoint_sha256'], 'teacher_fingerprint': teacher['fingerprint'],
        'teacher_source_sha256': teacher['runtime_source_sha256'], 'teacher_raw_dev_macro_f1': receipt['raw_dev_macro_f1'],
        'prepared_identity': job['prepared_identity'], 'label_count': cfg.label_count,
        'cache_sha256': sha256_file(destination), 'training_only': True,
        'teacher_foundation_parameters': receipt['parameter_report']['foundation_unique_parameters'],
        'student_foundation_parameters': job['parent_parameter_count'], 'test_labels_used': False,
        'teacher_loader_sha256': sha256_file(Path(__file__))}
    atomic_json(root / 'teacher_reference.json', reference)
    return reference


def preflight(cfg, catalog):
    """Find missing inputs before the several-minute student tokenization pass."""
    from dataclasses import replace
    from lsi.preparation import prepared_input
    prepared = prepared_input(cfg)
    manifest = _read(prepared / 'prepared_manifest.json')
    cfg = replace(cfg, label_count=_read(prepared / 'label_schema.json')['label_count'])
    cases = {}
    for split in ('train', 'dev'):
        cases[split] = []
        with (prepared / (split + '.jsonl')).open(encoding='utf-8') as stream:
            for line in stream:
                if line.strip():
                    row = json.loads(line)
                    cases[split].append({'id': row['id'], 'gold': row['gold']})
    job = {'prepared_identity': manifest['identity'], 'cache': {'cases': cases}}
    for path in (cfg.run_dir, cfg.resume_run_dir):
        if path and _reuse(path, manifest['identity'], cfg.label_count, len(cases['train'])):
            print('Verified existing KD teacher cache; separate teacher attachment is unnecessary for this resume.', flush=True)
            return
    selected = select_teacher(job, cfg, catalog)
    print('Teacher preflight passed:', selected[1], '; no student updates have run yet.', flush=True)


''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('p5_teacher_compat.py', []).append(r'''def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    parser.add_argument('--catalog', required=True)
    parser.add_argument('--preflight', action='store_true')
    args = parser.parse_args()
    from lsi.config import LSIConfig
    cfg = LSIConfig(**_read(args.plan)['config'])
    if args.preflight:
        preflight(cfg, _read(args.catalog))
        return
    import lsi.prepare as preparation
    preparation.teacher_inputs = teacher_inputs
    import lsi.workflow as workflow
    workflow.main()


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

Helper `sitecustomize.py`

In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('sitecustomize.py', []).append(r'''"""Run-local startup hook inherited by subprocesses and mp.spawn children.

Keep a pre-existing environment startup hook. This directory is inserted only
in the launch environment of this notebook, never installed into site-packages.
"""
import importlib.machinery
import importlib.util
import os
from pathlib import Path
import sys
import traceback


def _original_startup_hook() -> None:
    own_directory = Path(__file__).resolve().parent
    other_paths = [entry for entry in sys.path
                   if Path(entry or os.getcwd()).resolve() != own_directory]
    spec = importlib.machinery.PathFinder.find_spec('sitecustomize', other_paths)
    if spec is None or spec.loader is None:
        return
    if spec.origin and Path(spec.origin).resolve() == Path(__file__).resolve():
        return
    original = importlib.util.module_from_spec(spec)
    current = sys.modules.get('sitecustomize')
    sys.modules['sitecustomize'] = original
    try:
        spec.loader.exec_module(original)
    finally:
        if current is not None:
            sys.modules['sitecustomize'] = current
        else:
            sys.modules.pop('sitecustomize', None)


try:
    _original_startup_hook()
    if os.environ.get('CS781_NUMPY_RESUME_REQUIRED') == '1':
        from cs781_numpy_resume import register_numpy_scalars
        register_numpy_scalars()
except Exception:
''')


In [ ]:
# [Code Cell]
TEACHER_FIX_PARTS.setdefault('sitecustomize.py', []).append(r'''    traceback.print_exc()
    if os.environ.get('CS781_NUMPY_RESUME_REQUIRED') == '1':
        sys.stderr.write('Checkpoint compatibility startup failed; training was not started.\n')
        sys.stderr.flush()
        os._exit(78)
''')


In [ ]:
# [Code Cell]
TEACHER_FIX_SHA256 = {'cs781_native_contracts.py': 'f76cae1b17467620d65bcdd85d0aba6d15b5bf8cf7f8e4a8b30580b7951e368c', 'cs781_numpy_resume.py': '5e82ff896f53b50f6bbe320503da0d0ef794368b99900bb4b61c3f835d82d17a', 'cs781_source_identity.py': 'bff746c41ac050d03b1b02aee9d6e709c334697ad9ff6e69af4edacd0923690c', 'p5_teacher_compat.py': 'cc47cfc693f57a0867850c8b268f77c45ec8d15cb13eed859d87ab5dd93667d7', 'sitecustomize.py': '08ca4d65105de3acc37d18d6f3c24580927a6ab076c2814c2287774081750172'}
compat_directory = code_directory / 'teacher_input_compat'
compat_directory.mkdir(exist_ok=True)
for name, parts in TEACHER_FIX_PARTS.items():
    text = ''.join(parts)
    assert hashlib.sha256(text.encode()).hexdigest() == TEACHER_FIX_SHA256[name], name
    compile(text, name, 'exec')
    (compat_directory / name).write_bytes(text.encode())
sys.path.insert(0, str(compat_directory))
import cs781_numpy_resume
cs781_numpy_resume.register_numpy_scalars()
print('Numeric checkpoint metadata compatibility:', cs781_numpy_resume.self_test())


In [ ]:
# [Code Cell]
PARENT_CATALOG={'a': {'family': 'a', 'runtime_version': 'cs781-model-a-v1', 'source_sha256': {'config.py': 'c0d1e9e7eb49a11e12ff0d7014cd07c6b90149460da5db3b06e9af71fbb3f390', 'model.py': '05118a8c9eec3e58adb272554e7356e81fb28c95cbe7905f39ebd5ee2d670319', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': '723193f75a39d3b8f16af1e6f1bfbd507b1775a0cd2430583d211e54e69d63c2', 'train.py': '3ea14c27742ab82e686083a669823678529d0f8beb46b2e6b2113f0aef452743', 'infer.py': '1cb8e96f2d49aa1c0b25c8e0ef6420c6d63f17ec8b72813d69b646748fb46013', 'sanity.py': '7716a61318767f9001ac08cdcafbe228f53c3c0d6f8fb9cc3806353ec2d6577f', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 2, 'audited_original_step': 29942, 'audited_original_fingerprint': '24f90bc6e148ffb53dcf6132138b69a95ced2e6a5ca34690601d29def059cddd', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalCausalDecoder'}, 'b': {'family': 'b', 'runtime_version': 'cs781-model-b-v1', 'source_sha256': {'config.py': '71407468d5c4b105b5db2c533fb736eb847b4b67cdf9d4f2ab014436adef1e9b', 'model.py': '09baae8e7ebe0508fb149a64c187975b7a86ae62376b5bc2ec4cef2fb2471a99', 'data.py': 'd01b01d43f5c4c24bb0429f7bbb8bbbd4c52d52c8787bd5ef3f40bdac28df919', 'state.py': '026036ddbdf43e949755da263eb97378855a28281129c6a63f3b3ea69abbe088', 'train.py': 'f928bb52eaafd17dc061d3003326fd13cf2f79b5ccbc226f6af0177429aefab0', 'infer.py': '80ee8fff95675dccb59cf19da091355020495c0f41ea16fdb17d8f24357a59f4', 'sanity.py': '6b1b96bc16a4d7f0945e139fd60f35813b0bf9d2b075c4fd893a87d432fc9212', '__init__.py': '3dfc8da18d4c5d73d0ff1b52dc572672f8095b605dcc86836832d77607749087'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 3, 'audited_original_step': 24267, 'audited_original_fingerprint': 'e686c0f98fb5fe8fc03553d0151961dd6af0dd52ac041753de0502521a922bb1', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalEncoderDecoder'}, 'c': {'family': 'c', 'runtime_version': 'cs781-model-c-v1', 'source_sha256': {'config.py': 'f9fea0bc87f74ccac761f7d4dc574c4d1315a459523dba21849279c17bc24ace', 'model.py': '6f91835c789d12aadb5355bc06050b8b6c1fc231cd16126fc39f31769706aad4', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'b3a3ed7109fdfe43ebc8e75db89f2916d33e95ebe901e99d3e4aa9b6e8422ee4', 'train.py': '9bf6cb023ad1c8fdc4fbb652b4b3b6d3551ae777df139184051f8d23f4e5c0c9', 'infer.py': '062010c87b66c7badcef8f7a62ef3a89fdc7f96149d317214b13a7e679f40b6a', 'sanity.py': 'c56b6ad692b90cb5afb23a03a75a26dde9d9b2c9ef3e14ca2ed37cb8b0f18a68', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '092deede43ffd17c00e5d11e1f4aa3a443b7939af4504c488426b70c53d1ed54', 'audited_original_epochs': 1, 'audited_original_step': 7383, 'audited_original_fingerprint': '8febd61da7f1d32f78eee1cd4a865d72e5a5b3a0068739210db323859c6d7caa', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalTransformerGRU'}, 'd': {'family': 'd', 'runtime_version': 'cs781-model-d-v1', 'source_sha256': {'config.py': '42bd096bdf40822a7cb1d84e7f437e3e64d2ef20b99ee81c884f60639378e63a', 'model.py': 'a2300d34524df50cb43a3ff700e72b3a8764f7566d66ba5065c048c5f115eae9', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'dee5a29320756f4b84ec349b24091180be65a8f4c86e7f16eb74343737f4e2db', 'train.py': 'efb53d8eb54a08ef1a4abc49d5437eb472ebf5047bf790cf2f041c17fd6cef37', 'infer.py': 'aa3b849b0095bbdb5d81d3bfafe6a8134d4481856ee7cb9db5fba2bd84b80b67', 'sanity.py': '8c02e1c1a1e521a5d1e83fd96b3cfba3fa9f7c76c5824f62d45297a4829ca298', '__init__.py': 'f49e11a2d96bae160f02704fd5818fe613e77534e06a0f16150dbc4bc2d4f5e6'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3a4dc2be3ff16945069b15d7a2e351012d4a0c9319b95b5e4534cc6596e8693f', 'audited_original_epochs': 1, 'audited_original_step': 1828, 'audited_original_fingerprint': 'b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalSparseDecoder'}}


In [ ]:
# [Code Cell]
from lsi.config import LSIConfig
from lsi.io_utils import atomic_json,read_json
SETTINGS=dict(experiment_id=EXPERIMENT_ID,family=FAMILY,method=METHOD,run_dir=OUTPUT_ROOT,
    prepared_dir=PREPARED_DIR,parent_run_dir=PARENT_RUN_DIR,tokenizer_path=TOKENIZER_PATH,
    metadata_path=METADATA_PATH,resume_run_dir=RESUME_RUN_DIR,teacher_run_dir=TEACHER_RUN_DIR,
    expected_parent_sha256=EXPECTED_PARENT_SHA256,max_epochs=MAX_EPOCHS,target_total_epochs=TARGET_TOTAL_EPOCHS,
    max_run_seconds=int(TRAINING_MINUTES*60),chunk_tokens=CHUNK_TOKENS,chunk_micro_batch=CHUNK_MICRO_BATCH,
    backbone_lr=BACKBONE_LR,head_lr=HEAD_LR,head_width=HEAD_WIDTH,dropout=DROPOUT,
    prompt_tokens=PROMPT_TOKENS,loss_type=LOSS_TYPE,export_teacher_logits=EXPORT_TRAIN_LOGITS)
if RESUME_RUN_DIR:
    previous=read_json(Path(RESUME_RUN_DIR)/'job.json')
    requested=SETTINGS.copy()
    if previous.get('version')!='cs781-lsi-v1' or previous['config']['experiment_id']!=EXPERIMENT_ID:
        raise ValueError('Resume must be this same Phase 5 experiment ID; use the exact matching notebook')
    SETTINGS=previous['config'].copy()
    transport={'run_dir','prepared_dir','resume_run_dir','parent_run_dir','tokenizer_path','metadata_path',
        'teacher_run_dir','target_total_epochs','max_run_seconds','export_teacher_logits'}
    SETTINGS.update({k:v for k,v in requested.items() if k in transport})
    print('Inherited saved learning settings, label dimension and schedule for exact recovery.')
LSIConfig(**SETTINGS).validate()
plan_path=Path(OUTPUT_ROOT)/'pipeline_plan.json'
catalog_path=code_directory/'parent_catalog.json'
atomic_json(plan_path,{'mode':MODE,'config':SETTINGS})
atomic_json(catalog_path,PARENT_CATALOG)
print(json.dumps(SETTINGS,indent=2))


[Markdown Cell]

## 5. Stream progress and request a safe stop

In [ ]:
# [Code Cell]
import codecs
import select
import signal

def run_streamed(command):
    environment=os.environ.copy()
    environment['PYTHONPATH']=os.pathsep.join((str(compat_directory), str(code_directory), environment.get('PYTHONPATH','')))
    environment['CS781_NUMPY_RESUME_REQUIRED']='1'
    process=subprocess.Popen(command,cwd=code_directory,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=0)
    decoder=codecs.getincrementaldecoder('utf-8')(errors='replace')
    assert process.stdout is not None
    try:
        while True:
            try:
                ready,_,_=select.select([process.stdout],[],[],1.0)
                if ready:
                    chunk=os.read(process.stdout.fileno(),8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk));sys.stdout.flush()
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                print('Requesting a saved stop after the current update. Keep this cell running.')
                process.send_signal(signal.SIGINT)
        status=process.wait()
        sys.stdout.write(decoder.decode(b'',final=True));sys.stdout.flush()
        if status:
            raise subprocess.CalledProcessError(status,command)
    finally:
        process.stdout.close()


[Markdown Cell]

## 6. Verify the actual inputs, train or resume, evaluate and export

[Markdown Cell]

### Verify teacher inputs before student tokenization
This check fails early with exact missing files/rejection reasons. If no teacher output is attached,
add it; a code change cannot invent the teacher targets required by knowledge distillation.
An existing valid saved teacher cache is reused when resuming this same KD experiment.

In [ ]:
# [Code Cell]
run_streamed([sys.executable, '-u', '-m', 'p5_teacher_compat', '--preflight',
              '--plan', str(plan_path), '--catalog', str(catalog_path)])


In [ ]:
# [Code Cell]
run_streamed([sys.executable,'-u','-m','p5_teacher_compat','--plan',str(plan_path),'--catalog',str(catalog_path)])


[Markdown Cell]

## 7. Read completion status and download the exact deliverable

In [ ]:
# [Code Cell]
from IPython.display import FileLink,display
report_name='profile_report.json' if MODE=='profile' else 'final_report.json'
report=read_json(Path(OUTPUT_ROOT)/report_name)
print(json.dumps(report,indent=2))
if report.get('status')=='submission_verified':display(FileLink(report['zip_path']))
if (Path(OUTPUT_ROOT)/'observations.zip').exists():display(FileLink(str(Path(OUTPUT_ROOT)/'observations.zip')))
print('Save the COMPLETE output directory. observations.zip has reports, not model tensors.')


[Markdown Cell]

Submit only the printed **submission_*.zip** to Phase 5 after `submission_verified`. It contains exactly root prediction.json with one original test ID and a unique integer statute list per case. Submit all six distinct required methods separately; a comparison/winner copy does not create another method.

Keep the complete task output: checkpoints and .json receipts, job/config, code/lsi, parent_source, tokenizer, data cache, prediction logits/order, decision rule and reports. best.pt is validation-selected; slot_*.pt preserve latest optimizer progress. A paused_recoverable status is incomplete and needs the same notebook resumed. Real private F1 is unknown until Codabench evaluates the ZIP.